<a href="https://colab.research.google.com/github/saadi223/AI-based-blood_damage_model/blob/main/ShifaAssistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/shifa-knowledge-assistant")

# 1. Folders banao
for folder in [PROJECT_DIR, PROJECT_DIR / "documents", PROJECT_DIR / "knowledge_base"]:
    folder.mkdir(parents=True, exist_ok=True)

# 2. Khaali Python/README files banao (baad ke phases mein bharenge)
for name in ["app.py", "build_index.py", "rag_pipeline.py", "README.md"]:
    (PROJECT_DIR / name).touch(exist_ok=True)

# 3. requirements.txt = project ko chahiye wali libraries
(PROJECT_DIR / "requirements.txt").write_text(
"""streamlit
pymupdf
python-docx
pandas
numpy
sentence-transformers
faiss-cpu
groq
""")

# 4. .gitignore = woh files jo GitHub par kabhi upload nahi honi chahiye
(PROJECT_DIR / ".gitignore").write_text(
"""# Secrets - NEVER upload these
.env
.streamlit/secrets.toml

# Python / Colab clutter
__pycache__/
*.pyc
.ipynb_checkpoints/
""")

# 5. Dikhao ki kya-kya bana
print("Project folder:", PROJECT_DIR)
for item in sorted(PROJECT_DIR.rglob("*")):
    print(("📁 " if item.is_dir() else "📄 ") + str(item.relative_to(PROJECT_DIR)))

Project folder: /content/drive/MyDrive/shifa-knowledge-assistant
📄 .gitignore
📄 README.md
📄 app.py
📄 build_index.py
📁 documents
📄 documents/ADM-01_Patient_Registration_Procedure.docx
📄 documents/ADM-02_Inpatient_Admission_and_Discharge_Procedure.txt
📄 documents/BF-01_Billing_Policy.pdf
📄 documents/BF-02_Accepted_Payment_Methods.csv
📄 documents/BF-03_Insurance_and_Refund_Policy.docx
📄 documents/BF-04_Room_and_Bed_Charges.csv
📄 documents/EM-01_Emergency_Admission_Policy.pdf
📄 documents/EM-02_Triage_Guidelines.docx
📄 documents/NR-01_Nursing_Handover_Procedure.docx
📄 documents/NR-02_Vital_Signs_Monitoring_Policy.pdf
📄 documents/PH-01_Medication_Dispensing_Policy.pdf
📄 documents/PH-02_High_Alert_Medication_Guidelines.txt
📄 documents/PH-03_Pharmacy_Counter_Timings.csv
📄 documents/PS-01_Patient_Identification_Policy.pdf
📄 documents/PS-02_Infection_Prevention_Guidelines.docx
📄 documents/PS-03_Fall_Prevention_Policy.txt
📄 evaluation_questions.csv
📄 hospital_documents.zip
📁 knowledge_base
📄 rag_

In [3]:
!pip install -q -r /content/drive/MyDrive/shifa-knowledge-assistant/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 40.6 MB/s eta 0:00:00


In [4]:
import importlib

packages = {
    "PyMuPDF": "fitz",
    "python-docx": "docx",
    "pandas": "pandas",
    "numpy": "numpy",
    "sentence-transformers": "sentence_transformers",
    "faiss-cpu": "faiss",
    "groq": "groq",
    "streamlit": "streamlit",
}

all_ok = True
for pip_name, import_name in packages.items():
    try:
        importlib.import_module(import_name)
        print(f"✅ {pip_name}")
    except ImportError:
        print(f"❌ {pip_name} - not installed")
        all_ok = False

print("\nAll good! Ready for Phase 2." if all_ok else "\nSomething is missing - send me the output.")

✅ PyMuPDF
✅ python-docx
✅ pandas
✅ numpy
✅ sentence-transformers
✅ faiss-cpu
✅ groq
✅ streamlit

All good! Ready for Phase 2.


# Phase 2

In [5]:
!pip install -q pymupdf python-docx pandas

from google.colab import drive
drive.mount('/content/drive')

import csv, os, html, zipfile
from pathlib import Path
import fitz                      # PyMuPDF
from docx import Document
from docx.shared import Pt, RGBColor
import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/shifa-knowledge-assistant")
DOCS_DIR = PROJECT_DIR / "documents"
DOCS_DIR.mkdir(parents=True, exist_ok=True)

DISCLAIMER = "DEMO / SYNTHETIC HOSPITAL DATA — NOT FOR REAL CLINICAL USE"
DISCLAIMER_ASCII = "DEMO / SYNTHETIC HOSPITAL DATA - NOT FOR REAL CLINICAL USE"
HOSPITAL = "SHIFA General Hospital (Fictional)"


def meta_line(doc):
    return (f"Document ID: {doc['id']} | Department: {doc['department']} | "
            f"Version: {doc['version']} | Effective: {doc['effective']} | "
            f"Owner: {doc['owner']}")


# ---------- TXT ----------
def write_txt(doc, path):
    lines = [DISCLAIMER, "=" * 70, HOSPITAL.upper(), doc["title"].upper(), "=" * 70,
             f"Document ID: {doc['id']}", f"Department: {doc['department']}",
             f"Version: {doc['version']}", f"Effective Date: {doc['effective']}",
             f"Owner: {doc['owner']}"]
    for heading, items in doc["sections"]:
        lines += ["", heading.upper(), "-" * len(heading)]
        for it in items:
            lines.append(it)
            if not it.startswith("- "):
                lines.append("")
    lines += ["=" * 70, DISCLAIMER]
    path.write_text("\n".join(lines), encoding="utf-8")


# ---------- DOCX ----------
def write_docx(doc, path):
    d = Document()
    sec = d.sections[0]
    hp = sec.header.paragraphs[0]
    hr = hp.add_run(DISCLAIMER)
    hr.bold = True
    hr.font.size = Pt(8)
    hr.font.color.rgb = RGBColor(0xB0, 0x00, 0x00)
    sec.footer.paragraphs[0].text = f"{HOSPITAL} | {doc['id']} v{doc['version']}"

    d.add_heading(doc["title"], level=0)
    p = d.add_paragraph()
    r = p.add_run(DISCLAIMER)
    r.bold = True
    r.font.color.rgb = RGBColor(0xB0, 0x00, 0x00)
    d.add_paragraph(meta_line(doc))
    for heading, items in doc["sections"]:
        d.add_heading(heading, level=1)
        for it in items:
            if it.startswith("- "):
                d.add_paragraph(it[2:], style="List Bullet")
            else:
                d.add_paragraph(it)
    d.save(str(path))


# ---------- PDF ----------
def write_pdf(doc, path):
    esc = html.escape
    parts = [f"<h1>{esc(doc['title'])}</h1>",
             f"<p><b>{esc(HOSPITAL)}</b></p>",
             f"<p style='color:#b00000'><b>{esc(DISCLAIMER)}</b></p>",
             f"<p style='font-size:9pt'>{esc(meta_line(doc))}</p>"]
    for heading, items in doc["sections"]:
        parts.append(f"<h2>{esc(heading)}</h2>")
        in_list = False
        for it in items:
            if it.startswith("- "):
                if not in_list:
                    parts.append("<ul>")
                    in_list = True
                parts.append(f"<li>{esc(it[2:])}</li>")
            else:
                if in_list:
                    parts.append("</ul>")
                    in_list = False
                parts.append(f"<p>{esc(it)}</p>")
        if in_list:
            parts.append("</ul>")
    html_text = "".join(parts)
    css = "body{font-family:sans-serif;font-size:11pt;} h1{font-size:18pt;} h2{font-size:13pt;}"

    story = fitz.Story(html=html_text, user_css=css)
    writer = fitz.DocumentWriter(str(path))
    mediabox = fitz.paper_rect("a4")
    where = mediabox + (50, 60, -50, -60)
    more = 1
    while more:
        dev = writer.begin_page(mediabox)
        more, _ = story.place(where)
        story.draw(dev)
        writer.end_page()
    writer.close()

    # Har page ke neeche disclaimer + page number
    pdf = fitz.open(str(path))
    total = len(pdf)
    for i, page in enumerate(pdf, start=1):
        page.insert_text((50, 820),
                         f"{DISCLAIMER_ASCII} | {doc['id']} | Page {i} of {total}",
                         fontsize=7, color=(0.7, 0, 0))
    tmp = path.with_suffix(".tmp.pdf")
    pdf.save(str(tmp))
    pdf.close()
    os.replace(tmp, path)


# ---------- CSV ----------
def write_csv(doc, path):
    with open(path, "w", newline="", encoding="utf-8") as f:
        f.write(f"# {DISCLAIMER}\n")
        f.write(f"# Title: {doc['title']}\n")
        f.write(f"# {meta_line(doc)}\n")
        w = csv.writer(f)
        w.writerow(doc["columns"])
        w.writerows(doc["rows"])


WRITERS = {"pdf": write_pdf, "docx": write_docx, "txt": write_txt, "csv": write_csv}
print("✅ Writer functions ready")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Writer functions ready


In [6]:
DOCS_A = [

# ------------------------------------------------------------------ EM-01
{
"id": "EM-01", "filename": "EM-01_Emergency_Admission_Policy.pdf",
"title": "Emergency Admission Policy", "department": "Emergency", "format": "pdf",
"version": "3.2", "effective": "01 January 2026", "owner": "Head of Emergency Department",
"sections": [
 ("1. Purpose and Scope", [
  "This policy explains how patients who arrive at the SHIFA General Hospital Emergency Department (ED) are received, registered, assessed and, when needed, admitted. It applies to all ED doctors, nurses, registration clerks, security staff and the Bed Management team.",
  "The ED is open 24 hours a day, 7 days a week. The ED direct extension is 1000."]),
 ("2. Guiding Principles", [
  "- Assessment and stabilisation come first. Paperwork, identity documents, deposits and insurance approvals must never delay life-saving treatment.",
  "- No patient is refused initial assessment because of inability to pay, a missing CNIC or a missing insurance approval.",
  "- Medico-legal or police formalities never delay treatment."]),
 ("3. Emergency Admission Procedure", [
  "Step 1 - Arrival and triage: Every patient is seen by the triage nurse within 5 minutes of arrival and given a triage category from 1 to 5 as described in EM-02 Triage Guidelines. Ambulance patients are received at the ambulance bay by an ED nurse. Category 1 patients go directly to the resuscitation bay.",
  "Step 2 - Quick registration: The ED registration clerk creates an emergency MRN using minimum data: name (or an alias such as Unknown-Male-01 if the patient cannot be identified), approximate age, sex, mode of arrival and a relative's contact number. Full registration under ADM-01 is completed within 24 hours by the Admissions Desk.",
  "Step 3 - Doctor assessment: The ED doctor assesses the patient within the time target of the triage category (see EM-02).",
  "Step 4 - Consent: Written consent for treatment is taken from the patient or guardian. If the patient cannot give consent and no relative is present, two physicians (one of them a senior doctor) document an emergency consent in the chart before life-saving treatment begins.",
  "Step 5 - Disposition decision: The ED physician decides between (a) discharge with written instructions, (b) ED observation for up to 6 hours, or (c) inpatient admission.",
  "Step 6 - Admission request: For admission, the ED physician contacts the on-call specialty consultant, completes the Admission Order and Admission Note, and sends a bed request to Bed Management (extension 1150). The target is to allocate a bed within 60 minutes of the admission decision. If no ward bed is free, the patient stays in the ED under ED nursing care and Bed Management escalates to the Duty Manager after 60 minutes.",
  "Step 7 - Identification and handover: An ID wristband is applied before the patient leaves the ED (see PS-01). The ED nurse gives an SBAR handover to the ward nurse (see NR-01).",
  "Step 8 - Financial counselling: The billing counsellor meets the family only after the patient is stabilised. The admission deposit may be deferred for up to 24 hours after stabilisation (see BF-01)."]),
 ("4. Medico-Legal Cases (MLC)", [
  "Road traffic accidents, assault, firearm injuries, burns, suspected poisoning and suspected abuse are flagged as MLC by the triage nurse or treating doctor. The Medico-Legal Officer and hospital security are informed. The patient is treated first; police formalities are completed in parallel. MLC papers are kept in a separate confidential register."]),
 ("5. Transfer to Another Hospital", [
  "If the required service is not available at SHIFA, the treating consultant decides on transfer only after stabilisation. A physician-to-physician handover is mandatory, the patient travels in an ambulance accompanied by a nurse, and a Transfer Form is completed. The patient or guardian is informed and gives consent unless the delay would be unsafe."]),
 ("6. Mass Casualty Incident", [
  "In a mass casualty incident the ED in-charge or Duty Manager activates Code Red by calling extension 1999. Elective ED activity pauses and the hospital disaster plan takes over."]),
]},

# ------------------------------------------------------------------ EM-02
{
"id": "EM-02", "filename": "EM-02_Triage_Guidelines.docx",
"title": "Triage Guidelines", "department": "Emergency", "format": "docx",
"version": "2.4", "effective": "01 January 2026", "owner": "Nurse Manager - Emergency",
"sections": [
 ("1. Purpose", [
  "Triage means sorting patients by how urgently they need care, not by the order of arrival. Triage is performed by a registered nurse with triage training, at the triage area next to the ED entrance. The triage nurse does not make a diagnosis."]),
 ("2. Triage Categories", [
  "SHIFA uses a five-level scale:",
  "- Category 1 - Resuscitation (Red): immediate threat to life, for example cardiac arrest, blocked airway or severe breathing difficulty. Seen immediately.",
  "- Category 2 - Emergent (Orange): high-risk situation, for example chest pain of suspected cardiac origin, stroke symptoms or severe pain (8 to 10 out of 10). Seen within 10 minutes.",
  "- Category 3 - Urgent (Yellow): stable patient who needs prompt assessment, for example moderate abdominal pain or a fracture with stable vital signs. Seen within 30 minutes.",
  "- Category 4 - Semi-urgent (Green): minor injuries or illness, for example a small cut or mild vomiting. Seen within 60 minutes.",
  "- Category 5 - Non-urgent (Blue): minor problems, for example a mild cold or a routine dressing. Seen within 120 minutes."]),
 ("3. Triage Assessment", [
  "The triage nurse must start the assessment within 5 minutes of the patient's arrival and aims to finish it in about 3 minutes. The nurse records: arrival time, chief complaint, heart rate, blood pressure, respiratory rate, oxygen saturation (SpO2), temperature, pain score (0 to 10), Glasgow Coma Scale (GCS) and known allergies.",
  "Any patient with chest pain must have an ECG within 10 minutes of arrival."]),
 ("4. Upgrade Rules", [
  "The triage category must be raised to at least Category 2 in these situations:",
  "- Oxygen saturation (SpO2) below 90 percent.",
  "- Systolic blood pressure below 90 mmHg with dizziness or fainting.",
  "- GCS below 13.",
  "- Infant under 3 months with fever of 38.0 degrees C or higher.",
  "- Suspected stroke (a stroke alert is also raised)."]),
 ("5. Re-triage of Waiting Patients", [
  "Category 3 patients who are still waiting are re-assessed every 30 minutes. Category 4 and 5 patients are re-assessed every 60 minutes. Any patient whose condition worsens is re-triaged immediately. Families are told to inform the triage nurse if the patient gets worse."]),
 ("6. Special Groups", [
  "- Children under 12 years are assessed using paediatric vital sign ranges.",
  "- Pregnant women with vaginal bleeding or abdominal pain are triaged at Category 3 or higher.",
  "- Patients with suspected infectious disease (for example fever with rash) are given a mask and moved to the isolation area as per PS-02."]),
 ("7. Documentation and Escalation", [
  "Every triage decision is recorded on the Triage Form with the time. If the triage nurse and the family disagree about urgency, or the nurse is unsure, the ED physician on duty is called to decide."]),
]},

# ------------------------------------------------------------------ ADM-01
{
"id": "ADM-01", "filename": "ADM-01_Patient_Registration_Procedure.docx",
"title": "Patient Registration Procedure", "department": "Admissions", "format": "docx",
"version": "2.1", "effective": "15 January 2026", "owner": "Manager - Patient Services",
"sections": [
 ("1. Purpose and Scope", [
  "Every patient at SHIFA General Hospital has exactly one Medical Record Number (MRN) for life. The MRN format is SH- followed by six digits (for example SH-004512). This procedure applies to OPD, inpatient and Admissions Desk staff."]),
 ("2. Counters and Timings", [
  "- OPD registration counters: Monday to Saturday 7:30 AM to 8:00 PM, Sunday 9:00 AM to 2:00 PM.",
  "- Admissions Desk (extension 1100): open 24 hours a day.",
  "- Emergency registration: 24 hours a day at the ED (see EM-01)."]),
 ("3. Documents Required", [
  "- Adult patients: original CNIC. Overseas Pakistanis may use NICOP and foreigners must show a passport.",
  "- Children under 18: the child's B-Form and the CNIC of a parent or guardian.",
  "- A mobile phone number for the patient or guardian.",
  "- Insurance or corporate panel card and, if applicable, the employer letter.",
  "- Referral letter, if the patient was referred by another doctor.",
  "Photocopies are not required because staff scan the originals."]),
 ("4. Registration Steps", [
  "1. Ask whether it is the patient's first visit.",
  "2. Search the system for an existing MRN. Search by CNIC number first, then mobile number, then name plus date of birth.",
  "3. If a record exists, verify and update the details. Do not create a new record.",
  "4. If no record exists, create a new one. Enter the name exactly as on the CNIC, date of birth, sex, address, phone number and next of kin.",
  "5. Record the payer type: self-pay, insurance panel or corporate.",
  "6. The patient signs the consent-to-treat form and the privacy notice acknowledgement.",
  "7. Issue the patient card showing the MRN. Admitted patients also receive an ID wristband (see PS-01).",
  "Time targets: 5 minutes for a returning patient and 10 minutes for a new patient."]),
 ("5. Special Situations", [
  "- Unconscious or unidentified patients are registered with an alias such as Unknown-Male-01. When the identity is confirmed, only the Health Information Management (HIM) team may merge the records.",
  "- If a duplicate MRN is discovered, do not create another one. Report it to HIM (extension 1120) on the same day. HIM merges the records within 3 working days.",
  "- Newborns are registered as Baby of (mother's name) with their own MRN within 24 hours of birth.",
  "- Patients who request privacy are flagged Restricted Access."]),
 ("6. Data Privacy", [
  "Staff may open a patient's record only when they need it for their job. Sharing patient information through WhatsApp or personal email is not allowed."]),
]},

# ------------------------------------------------------------------ ADM-02
{
"id": "ADM-02", "filename": "ADM-02_Inpatient_Admission_and_Discharge_Procedure.txt",
"title": "Inpatient Admission and Discharge Procedure", "department": "Admissions", "format": "txt",
"version": "1.8", "effective": "01 February 2026", "owner": "Manager - Patient Services",
"sections": [
 ("A. Planned Inpatient Admission", [
  "1. The doctor writes the Admission Order and completes the Admission Form.",
  "2. The Admissions Desk (extension 1100) confirms registration and payer type and collects the deposit as per BF-01.",
  "3. Bed Management (extension 1150) allocates a bed.",
  "4. A porter escorts the patient to the ward.",
  "5. The ward nurse completes the admission nursing assessment within 1 hour of the patient reaching the ward. The ID wristband and allergy status are checked (see PS-01).",
  "6. The admitting doctor documents the initial assessment within 24 hours of admission. Emergency admissions are assessed by the ED doctor before transfer.",
  "7. The nurse orients the patient and family: call bell, visiting rules and belongings list. Valuables are handed to the family."]),
 ("B. Visiting Hours", [
  "- General wards: every day from 4:00 PM to 7:00 PM.",
  "- One attendant may stay with each patient 24 hours a day with an attendant pass.",
  "- ICU and CCU: every day from 5:00 PM to 6:00 PM, maximum 2 visitors at a time, 10 minutes each.",
  "- Children under 12 years are not allowed in ICU or wards for infection control reasons."]),
 ("C. Discharge Procedure", [
  "1. The doctor writes the discharge order and discharge summary. For planned discharges the order must be written by 10:00 AM.",
  "2. The nurse does medication reconciliation with the pharmacy. Discharge medicines are dispensed at the Discharge Medication Counter (8:00 AM to 8:00 PM). Outside these hours the Inpatient Pharmacy dispenses them.",
  "3. The final bill is prepared within 2 hours of the discharge order (see BF-01). The attendant clears dues at the billing counter, extension 1700.",
  "4. The nurse gives discharge counselling: medicines, diet, warning signs and the follow-up appointment date.",
  "5. The patient leaves the ward with wheelchair assistance. Checkout time is 12:00 noon. The target is that the patient leaves within 3 hours of the discharge order."]),
 ("D. Leaving Against Medical Advice (LAMA)", [
  "If a patient wishes to leave against medical advice, the treating doctor explains the risks. The patient or guardian signs the LAMA form in front of a witness. The hospital still provides a discharge summary and the medicines needed for a safe transition. Dues are settled as usual."]),
]},

# ------------------------------------------------------------------ PS-01
{
"id": "PS-01", "filename": "PS-01_Patient_Identification_Policy.pdf",
"title": "Patient Identification Policy", "department": "Patient Safety", "format": "pdf",
"version": "2.0", "effective": "01 March 2026", "owner": "Patient Safety Officer",
"sections": [
 ("1. Purpose", [
  "Correct patient identification prevents wrong-patient errors in medication, blood transfusion, tests and procedures. This policy applies to all clinical staff. The Patient Safety Office is reachable on extension 1400."]),
 ("2. The Two-Identifier Rule", [
  "Before any care activity, staff must confirm at least two identifiers: (1) the patient's full name and (2) the MRN or the date of birth. Room number, bed number or diagnosis must never be used as an identifier.",
  "Ask the patient to state their own name and date of birth. Do not ask a leading question such as 'Are you Mr Khan?'. If the patient cannot speak, check the wristband and confirm with the attendant."]),
 ("3. ID Wristbands", [
  "- White band: standard identification (name, MRN, date of birth, sex) for every inpatient and every ED patient who is admitted or kept for observation.",
  "- Red band: allergy alert.",
  "- Yellow band: fall risk (see PS-03).",
  "- Purple band: Do Not Resuscitate (DNR) order.",
  "The band is applied before the patient leaves the ED or the Admissions Desk. If a band is missing, unreadable or wrong, stop, verify identity using two identifiers and replace the band immediately."]),
 ("4. When Identification Must Be Checked", [
  "- Before giving any medication.",
  "- Before a blood transfusion.",
  "- Before collecting blood or any other specimen.",
  "- Before any procedure, surgery or radiology test.",
  "- Before transporting the patient to another department."]),
 ("5. Blood Transfusion and Specimens", [
  "A blood transfusion needs two licensed staff members (two nurses, or a nurse and a doctor) to check the patient identity and the blood bag details together at the bedside. Specimen labels are written at the bedside, in front of the patient, straight after collection. Labelling tubes in advance is prohibited."]),
 ("6. Newborns", [
  "Two identical ID bands showing the mother's name, mother's MRN, baby's sex and date and time of birth are applied before the baby leaves the delivery room: one on the wrist and one on the ankle."]),
 ("7. Patients With Similar Names", [
  "A Name Alert sticker is placed on the chart and the bed board. Where possible, patients with similar names are not placed in the same room."]),
 ("8. Reporting", [
  "Any identification error or near miss must be reported in the incident reporting system within 24 hours."]),
]},

# ------------------------------------------------------------------ PS-02
{
"id": "PS-02", "filename": "PS-02_Infection_Prevention_Guidelines.docx",
"title": "Infection Prevention Guidelines", "department": "Patient Safety", "format": "docx",
"version": "3.0", "effective": "01 March 2026", "owner": "Infection Control Nurse",
"sections": [
 ("1. Purpose", [
  "These guidelines protect patients, visitors and staff from healthcare-associated infections. Standard precautions apply to all patients at all times. The Infection Control team is reachable on extension 1300."]),
 ("2. Hand Hygiene", [
  "Hand hygiene is performed at the Five Moments: before touching a patient, before a clean or aseptic procedure, after body fluid exposure risk, after touching a patient, and after touching the patient's surroundings.",
  "- Alcohol-based hand rub: rub all surfaces of the hands for 20 to 30 seconds until dry.",
  "- Soap and water: wash for 40 to 60 seconds when hands are visibly dirty or soiled with blood or body fluids, and after caring for patients with suspected Clostridioides difficile or norovirus.",
  "- Nails must be short. Artificial nails are not allowed."]),
 ("3. Personal Protective Equipment (PPE)", [
  "- Putting on (donning) order: gown, mask, eye protection, gloves.",
  "- Taking off (doffing) order: gloves, eye protection and gown, then mask. Perform hand hygiene immediately afterwards.",
  "- Gloves are changed between patients and are never washed or reused."]),
 ("4. Isolation Precautions", [
  "- Contact precautions (for example MRSA, C. difficile): single room or cohort area, gown and gloves.",
  "- Droplet precautions (for example influenza, meningococcal disease): surgical mask within 2 metres of the patient.",
  "- Airborne precautions (for example tuberculosis, measles, chickenpox): negative pressure room with the door closed and an N95 respirator for staff.",
  "An isolation sign is placed on the door. Isolated patients wear a mask during transport."]),
 ("5. Sharps Safety and Needle-Stick Injury", [
  "Never recap used needles. Discard sharps immediately at the point of use into a puncture-resistant sharps container. Replace the container when it is three-quarters full.",
  "If a needle-stick or sharps injury happens:",
  "1. Let the wound bleed and wash it with soap and running water. Do not squeeze or scrub. For splashes in the eyes or mouth, rinse with water for 10 minutes.",
  "2. Inform the nurse in-charge and go to the ED or Occupational Health within 2 hours for risk assessment and post-exposure prophylaxis if needed.",
  "3. Inform Infection Control (extension 1300) and file an incident report within 24 hours."]),
 ("6. Waste and Cleaning", [
  "- Yellow bag: infectious waste. Black bag: general waste. Never mix them.",
  "- Blood spills: cover with absorbent material, apply 0.5 percent chlorine solution and leave for 10 minutes before cleaning.",
  "- High-touch surfaces (bed rails, call bells, door handles) are cleaned at least twice a day and after every discharge."]),
 ("7. Outbreaks", [
  "If two or more linked cases of the same infection occur in a ward within 7 days, the ward nurse in-charge must notify the Infection Control Nurse within 24 hours."]),
]},

# ------------------------------------------------------------------ PS-03
{
"id": "PS-03", "filename": "PS-03_Fall_Prevention_Policy.txt",
"title": "Fall Prevention Policy", "department": "Patient Safety", "format": "txt",
"version": "1.5", "effective": "01 March 2026", "owner": "Patient Safety Officer",
"sections": [
 ("1. Purpose", [
  "This policy reduces the number of patient falls and the injuries caused by falls in all inpatient areas of SHIFA General Hospital."]),
 ("2. Fall Risk Assessment", [
  "Adult patients are assessed with the Morse Fall Scale on admission, after transfer to another ward, after any fall, after a significant change in condition, and at least once every shift. Children are assessed with the Humpty Dumpty scale.",
  "Morse Fall Scale interpretation:",
  "- Score 0 to 24: low risk.",
  "- Score 25 to 44: moderate risk.",
  "- Score 45 or above: high risk."]),
 ("3. Precautions for High-Risk Patients", [
  "- Yellow fall-risk wristband (see PS-01) and a Fall Risk sign at the bedside.",
  "- Bed kept in the lowest position, brakes on, call bell within reach.",
  "- Non-slip footwear.",
  "- Hourly rounding by nursing staff, including offering help to use the toilet.",
  "- Patient and family are taught to call for help before getting out of bed."]),
 ("4. Action After a Fall", [
  "1. Do not move the patient until the nurse has assessed for injury.",
  "2. Notify the doctor immediately. The doctor must examine the patient within 30 minutes.",
  "3. Check vital signs and neurological status every 15 minutes for the first hour, every 30 minutes for the next 2 hours, then every 4 hours for 24 hours.",
  "4. Inform the family.",
  "5. File an incident report within 24 hours and hold a post-fall team huddle within 24 hours to find the cause and prevent a repeat."]),
]},
]
print("✅ Part A ready:", len(DOCS_A), "documents")

✅ Part A ready: 7 documents


In [7]:
DOCS_B = [

# ------------------------------------------------------------------ PH-01
{
"id": "PH-01", "filename": "PH-01_Medication_Dispensing_Policy.pdf",
"title": "Medication Dispensing Policy", "department": "Pharmacy", "format": "pdf",
"version": "4.0", "effective": "01 April 2026", "owner": "Chief Pharmacist",
"sections": [
 ("1. Purpose", [
  "This policy ensures that every medicine dispensed at SHIFA General Hospital is correct, safe and properly labelled. It applies to all pharmacists and pharmacy technicians. The Pharmacy main extension is 1500."]),
 ("2. Valid Prescription", [
  "A medicine is dispensed only against a valid prescription or medication order that contains:",
  "- Patient name and MRN.",
  "- Age, and body weight for children.",
  "- Generic drug name, strength, dose, route, frequency and duration.",
  "- Prescriber name, signature and PMDC registration number.",
  "- Date of the prescription.",
  "Incomplete or unclear prescriptions are returned to the prescriber for clarification. Pharmacy staff must never guess."]),
 ("3. Pharmacist Verification", [
  "Before dispensing, the pharmacist checks: known allergies (see the red wristband in PS-01), correct dose for age and weight, drug interactions, and duplicate therapy. High-alert medicines follow PH-02."]),
 ("4. Labelling and Counselling", [
  "Every dispensed medicine is labelled with the patient name, MRN, drug name, strength, dose, frequency, expiry date and special instructions. The pharmacist counsels patients receiving discharge medicines and anyone receiving a medicine for the first time."]),
 ("5. Inpatient Dispensing", [
  "The Inpatient Pharmacy supplies wards with unit-dose medicines. Routine orders are delivered within 30 minutes of receiving the order and STAT (urgent) orders within 15 minutes."]),
 ("6. Controlled Drugs", [
  "Controlled drugs are dispensed only against a controlled drug prescription signed by a consultant. They are dispensed only by the Inpatient Pharmacy and the Emergency Pharmacy. They are stored in a double-lock cabinet. Two authorised persons count them at every shift change and sign the register. Any discrepancy is reported immediately to the Pharmacy In-charge."]),
 ("7. Verbal Orders", [
  "Verbal orders are accepted only in emergencies. The receiver writes the order down, reads it back to the doctor and the doctor signs it within 24 hours."]),
 ("8. Restricted Antibiotics", [
  "Restricted antibiotics are dispensed only after approval from the Infectious Disease consultant."]),
 ("9. Expired Medicines and Returns", [
  "Expired medicines are never dispensed. Items within 90 days of expiry are separated and marked. Once a medicine has left the pharmacy it cannot be returned to stock, except when the pharmacy made a dispensing error."]),
]},

# ------------------------------------------------------------------ PH-02
{
"id": "PH-02", "filename": "PH-02_High_Alert_Medication_Guidelines.txt",
"title": "High-Alert Medication Guidelines", "department": "Pharmacy", "format": "txt",
"version": "2.2", "effective": "01 April 2026", "owner": "Chief Pharmacist",
"sections": [
 ("1. What Are High-Alert Medications", [
  "High-alert medications carry a higher risk of serious harm if used incorrectly. At SHIFA General Hospital the following are classified as high-alert:",
  "- Insulin",
  "- Heparin",
  "- Concentrated potassium chloride",
  "- Magnesium sulfate",
  "- Opioids",
  "- Chemotherapy agents",
  "- Neuromuscular blocking agents"]),
 ("2. Independent Double-Check", [
  "Two licensed nurses must independently check the drug, dose, route, patient identity and pump settings before any high-alert medicine is administered. Both nurses sign the medication chart."]),
 ("3. Storage Rules", [
  "- Concentrated potassium chloride is not stocked in general wards. It is kept locked and labelled only in the ICU and the Emergency Department.",
  "- High-alert medicines carry a HIGH ALERT sticker.",
  "- Look-alike sound-alike (LASA) medicines are stored apart from each other and written using Tall Man lettering."]),
 ("4. Specific Rules", [
  "- Insulin: use only insulin syringes. Always write doses as the word units, never as U.",
  "- Heparin: use only the standard concentrations supplied by the pharmacy. Blood clotting tests (aPTT) are monitored as ordered.",
  "- Opioids: naloxone must be available in every ward."]),
 ("5. Error Reporting", [
  "Every medication error or near miss is reported within 24 hours. An error that causes harm is reported immediately to the Pharmacy In-charge and the Patient Safety Officer (extension 1400)."]),
]},

# ------------------------------------------------------------------ PH-03 (CSV)
{
"id": "PH-03", "filename": "PH-03_Pharmacy_Counter_Timings.csv",
"title": "Pharmacy Counter Timings", "department": "Pharmacy", "format": "csv",
"version": "1.3", "effective": "01 April 2026", "owner": "Chief Pharmacist",
"columns": ["counter_name", "location", "serves", "opening_time", "closing_time", "days", "extension", "notes"],
"rows": [
 ["Emergency Pharmacy", "Ground Floor, next to Emergency Department", "Emergency patients", "00:00", "23:59", "Daily (24 hours)", "1510", "Can dispense controlled drugs against a consultant-signed prescription"],
 ["Inpatient Pharmacy", "First Floor, Block A", "Wards, ICU and HDU", "00:00", "23:59", "Daily (24 hours)", "1520", "Unit-dose ward supply. Routine orders within 30 minutes, STAT orders within 15 minutes. Dispenses discharge medicines outside Discharge Counter hours"],
 ["OPD Pharmacy", "Ground Floor, OPD Block", "Outpatients", "08:00", "22:00", "Daily", "1530", "Does not dispense controlled drugs"],
 ["Discharge Medication Counter", "Ground Floor, near Main Billing Office", "Patients being discharged", "08:00", "20:00", "Daily", "1525", "Pharmacist counselling provided"],
 ["Central Pharmacy Store", "Basement", "Staff only (no patient dispensing)", "09:00", "17:00", "Monday to Saturday", "1540", "Receives supplier deliveries"],
]},

# ------------------------------------------------------------------ NR-01
{
"id": "NR-01", "filename": "NR-01_Nursing_Handover_Procedure.docx",
"title": "Nursing Handover Procedure", "department": "Nursing", "format": "docx",
"version": "2.3", "effective": "01 May 2026", "owner": "Chief Nursing Officer",
"sections": [
 ("1. Purpose", [
  "A structured handover makes sure that important patient information is not lost when one nurse takes over from another. It applies to all inpatient wards, the ED and the ICU."]),
 ("2. Shift Timings and Handover Times", [
  "- Morning shift: 7:00 AM to 3:00 PM.",
  "- Evening shift: 3:00 PM to 11:00 PM.",
  "- Night shift: 11:00 PM to 7:00 AM.",
  "Handover takes place at 7:00 AM, 3:00 PM and 11:00 PM and must be completed within 30 minutes. The outgoing nurse remains responsible for the patient until the incoming nurse accepts the handover."]),
 ("3. The SBAR Format", [
  "All handovers use SBAR:",
  "- Situation: patient name, MRN, bed, diagnosis and the reason for care.",
  "- Background: relevant history, allergies, code status (DNR), isolation status.",
  "- Assessment: latest vital signs and early warning score, pain, fall risk, pressure injury risk, IV lines and drains.",
  "- Recommendation: pending tests, medicines due in the next 2 hours, planned procedures and family concerns."]),
 ("4. Where and How", [
  "Handover is done at the bedside so both nurses can see the patient, check the ID wristband (see PS-01) and check lines and equipment. Sensitive information is discussed at the nursing station, not in front of other patients."]),
 ("5. Controlled Drug Count", [
  "At every handover the outgoing and incoming nurse count the controlled drugs together and both sign the register (see PH-01)."]),
 ("6. Documentation", [
  "The signed handover sheet is filed in the patient's nursing record. If a nurse is late or absent, the nurse in-charge must be told immediately and covers the handover."]),
 ("7. Transfers Between Departments", [
  "When a patient moves between departments (for example ED to ward) the sending nurse gives an SBAR handover to the receiving nurse, and the Transfer Checklist is completed before the patient leaves."]),
]},

# ------------------------------------------------------------------ NR-02
{
"id": "NR-02", "filename": "NR-02_Vital_Signs_Monitoring_Policy.pdf",
"title": "Vital Signs Monitoring Policy", "department": "Nursing", "format": "pdf",
"version": "2.0", "effective": "01 May 2026", "owner": "Chief Nursing Officer",
"sections": [
 ("1. Purpose", [
  "This policy sets how often vital signs are measured and how abnormal results must be escalated, so that a worsening patient is recognised early."]),
 ("2. Parameters", [
  "Each set of vital signs includes temperature, pulse, blood pressure, respiratory rate, oxygen saturation (SpO2), level of consciousness (AVPU) and a pain score from 0 to 10."]),
 ("3. Monitoring Frequency", [
  "- Stable ward patients: at least once every 8 hours (once per shift).",
  "- Post-operative patients: every 15 minutes for the first hour, every 30 minutes for the next 2 hours, hourly for the next 4 hours, then every 4 hours if stable.",
  "- Blood transfusion: before starting, 15 minutes after starting, then hourly and at the end of the transfusion.",
  "- ICU and HDU: continuous monitoring with charting every hour."]),
 ("4. Early Warning Score (EWS) and Response", [
  "- Score 0 to 4 (low risk): continue routine monitoring. If any single parameter scores 3, inform the nurse in-charge.",
  "- Score 5 or 6 (medium risk): inform the responsible doctor within 30 minutes and increase monitoring to at least hourly.",
  "- Score 7 or more (high risk): call the Rapid Response Team on extension 1777 immediately, start continuous monitoring and stay with the patient."]),
 ("5. Pain Assessment", [
  "Pain is assessed with every set of vital signs on a scale of 0 to 10. A score of 4 or more must be reported to the doctor or treated with the prescribed analgesic, and reassessed within 30 minutes."]),
 ("6. Documentation and Equipment", [
  "Readings are recorded immediately and no later than 30 minutes after measurement. Abnormal values are highlighted. Previous values must never be copied forward. Monitors and thermometers are calibrated every 6 months by Biomedical Engineering and faulty devices are tagged and removed."]),
 ("7. Children", [
  "Children under 12 years are monitored with the Paediatric Early Warning Score (PEWS), which uses age-specific ranges. The adult EWS chart is not used for them."]),
]},

# ------------------------------------------------------------------ BF-01
{
"id": "BF-01", "filename": "BF-01_Billing_Policy.pdf",
"title": "Billing Policy", "department": "Billing & Finance", "format": "pdf",
"version": "3.1", "effective": "01 June 2026", "owner": "Head of Finance",
"sections": [
 ("1. Purpose", [
  "This policy explains how SHIFA General Hospital charges for services, collects deposits and issues bills. The Billing Office main extension is 1700."]),
 ("2. Billing Office Hours", [
  "- Main Billing Office: every day from 8:00 AM to 8:00 PM.",
  "- Emergency Billing Counter: open 24 hours a day. Patients discharged after 8:00 PM settle dues here."]),
 ("3. Admission Deposits", [
  "A refundable deposit is collected at admission:",
  "- General Ward: PKR 20,000.",
  "- Semi-Private Room: PKR 35,000.",
  "- Private Room: PKR 50,000.",
  "- ICU, CCU and NICU: PKR 100,000.",
  "Deposits for other room types are listed in BF-04. Emergency admissions may defer the deposit for up to 24 hours after the patient is stabilised. No emergency patient is denied stabilising treatment because of inability to pay."]),
 ("4. Estimates and Bills", [
  "- A written cost estimate for a planned procedure is provided within 4 working hours of request.",
  "- An interim bill is shared with the attendant every 24 hours. When charges reach 80 percent of the deposit, the billing staff ask the attendant to top up the deposit.",
  "- The final bill is prepared within 2 hours of the discharge order."]),
 ("5. Room Charges and Checkout Time", [
  "Room charges are per day (see BF-04). Checkout time is 12:00 noon. A patient who leaves after 12:00 noon and up to 6:00 PM is charged a half-day room rate. A patient who leaves after 6:00 PM is charged a full day."]),
 ("6. Concessions", [
  "- Senior citizens aged 65 and above receive 10 percent off OPD consultation fees.",
  "- Hospital employees and their dependents receive 25 percent off non-consumable services (room, consultation and procedures). Medicines and implants are excluded.",
  "- Concessions cannot be combined and do not apply to insurance panel rates.",
  "- Patients who cannot afford treatment may apply to the Patient Welfare Committee through the Billing Supervisor."]),
 ("7. Billing Disputes", [
  "A patient or attendant who disagrees with a bill must raise the complaint with the Billing Supervisor (extension 1701) within 7 days of the final bill. The Billing Supervisor gives a decision within 3 working days."]),
]},

# ------------------------------------------------------------------ BF-02 (CSV)
{
"id": "BF-02", "filename": "BF-02_Accepted_Payment_Methods.csv",
"title": "Accepted Payment Methods", "department": "Billing & Finance", "format": "csv",
"version": "1.4", "effective": "01 June 2026", "owner": "Head of Finance",
"columns": ["payment_method", "accepted_at", "currency", "limit_per_transaction_pkr", "processing_fee", "notes"],
"rows": [
 ["Cash", "All billing counters and OPD Pharmacy", "PKR", "500000", "None", "Cash above PKR 500,000 must be paid by bank transfer or cheque"],
 ["Debit Card (Visa, Mastercard, UnionPay)", "All billing counters and pharmacies", "PKR", "No limit", "None", "PIN required"],
 ["Credit Card (Visa, Mastercard)", "All billing counters and pharmacies", "PKR", "No limit", "1.5 percent of transaction", "Fee is added to the bill"],
 ["Bank Transfer (IBFT / Raast)", "Main Billing Office only", "PKR", "No limit", "None", "Patient must show the transfer confirmation. Receipt is issued once the amount is credited"],
 ["Crossed Cheque", "Main Billing Office (pre-approved corporate accounts only)", "PKR", "No limit", "None", "Payable to SHIFA General Hospital. Needs Billing Supervisor approval"],
 ["Mobile Wallet (JazzCash, Easypaisa)", "Main Billing Office and OPD counters", "PKR", "100000", "None", "Show the payment confirmation message"],
 ["Insurance Panel (Cashless)", "Insurance Desk", "PKR", "As per pre-authorization", "None", "Requires pre-authorization. See BF-03"],
 ["Foreign Currency", "Not accepted", "-", "-", "-", "Payments are accepted in PKR only"],
]},

# ------------------------------------------------------------------ BF-03
{
"id": "BF-03", "filename": "BF-03_Insurance_and_Refund_Policy.docx",
"title": "Insurance and Refund Policy", "department": "Billing & Finance", "format": "docx",
"version": "2.5", "effective": "01 June 2026", "owner": "Head of Finance",
"sections": [
 ("1. Purpose", [
  "This policy explains how insured patients are billed and when payments are refunded. The Insurance Desk is on extension 1710."]),
 ("2. Cashless Facility for Panel Insurers", [
  "SHIFA offers cashless treatment for these panel insurers: Pak-Care Health Insurance, National Shield Insurance and Unity Corporate Health. Panel patients must show their insurance card and CNIC."]),
 ("3. Pre-Authorization", [
  "- Planned admissions: the insurance card and CNIC are sent to the Insurance Desk at least 1 working day before admission. The insurer normally replies within 4 working hours.",
  "- Emergency admissions: treatment is never delayed. The Insurance Desk informs the insurer within 24 hours of admission."]),
 ("4. Co-payment and Non-Covered Items", [
  "The patient pays a co-payment of 10 percent of the eligible bill unless the policy states otherwise. Items not covered by the policy (for example cosmetic procedures and personal comfort items) are paid by the patient at discharge. If the insurer rejects a claim, the patient is responsible for the full amount."]),
 ("5. Non-Panel Insurers", [
  "Patients with a non-panel insurer pay the bill themselves. The hospital provides the final bill, discharge summary and investigation reports within 2 working days so the patient can claim reimbursement."]),
 ("6. Refunds", [
  "- Unused deposit balance is refunded after the final bill is settled. Cash refunds up to PKR 20,000 are paid immediately at the billing counter. Amounts above PKR 20,000 are paid by bank transfer or crossed cheque within 7 working days.",
  "- Card payments are refunded to the same card within 10 working days.",
  "- OPD consultation fee: fully refunded if the consultation has not started, on the same day with the original receipt.",
  "- Laboratory and radiology tests: fully refunded if the sample has not been collected or the test has not started. No refund after that.",
  "- Medicines that have left the pharmacy are not refundable, except in case of a pharmacy dispensing error (see PH-01)."]),
 ("7. Refund Process", [
  "Every refund needs the original receipt and the CNIC of the person requesting it, and must be approved by the Billing Supervisor."]),
 ("8. Cancellation of Planned Procedures", [
  "If a planned surgery is cancelled with more than 24 hours notice, the full deposit is refunded. If it is cancelled with less than 24 hours notice, an OT booking fee of PKR 5,000 is non-refundable."]),
]},

# ------------------------------------------------------------------ BF-04 (CSV)
{
"id": "BF-04", "filename": "BF-04_Room_and_Bed_Charges.csv",
"title": "Room and Bed Charges", "department": "Billing & Finance", "format": "csv",
"version": "2.0", "effective": "01 June 2026", "owner": "Head of Finance",
"columns": ["room_type", "daily_charge_pkr", "deposit_required_pkr", "includes", "notes"],
"rows": [
 ["General Ward", "4500", "20000", "Bed, nursing care, basic meals", "Shared ward"],
 ["Semi-Private Room", "9000", "35000", "Bed, nursing care, meals, attached bathroom", "Two beds per room"],
 ["Private Room", "15000", "50000", "Bed, nursing care, meals, TV, attendant bed", "Single patient"],
 ["Deluxe Suite", "25000", "75000", "Bed, nursing care, meals, TV, attendant bed, sitting area", "Single patient"],
 ["High Dependency Unit (HDU)", "22000", "75000", "Bed, close monitoring, nursing care", "Medicines and procedures billed separately"],
 ["Intensive Care Unit (ICU)", "35000", "100000", "Bed, continuous monitoring, intensive nursing care", "Ventilator charges billed separately"],
 ["Coronary Care Unit (CCU)", "35000", "100000", "Bed, cardiac monitoring, intensive nursing care", "Procedures billed separately"],
 ["Neonatal ICU (NICU)", "40000", "100000", "Incubator or cot, monitoring, neonatal nursing care", "Ventilator charges billed separately"],
 ["Emergency Observation Bed", "3000", "0", "Bed and nursing care for up to 6 hours", "Charged per observation episode of up to 6 hours. No deposit required"],
]},
]

ALL_DOCS = DOCS_A + DOCS_B
print("✅ Part B ready:", len(DOCS_B), "documents | Total:", len(ALL_DOCS))

✅ Part B ready: 9 documents | Total: 16


In [8]:
for doc in ALL_DOCS:
    path = DOCS_DIR / doc["filename"]
    WRITERS[doc["format"]](doc, path)
    print(f"✅ {doc['format'].upper():4} | {doc['department']:18} | {doc['filename']}")

print("\nTotal files in documents/:", len([f for f in DOCS_DIR.iterdir() if f.is_file()]))

✅ PDF  | Emergency          | EM-01_Emergency_Admission_Policy.pdf
✅ DOCX | Emergency          | EM-02_Triage_Guidelines.docx
✅ DOCX | Admissions         | ADM-01_Patient_Registration_Procedure.docx
✅ TXT  | Admissions         | ADM-02_Inpatient_Admission_and_Discharge_Procedure.txt
✅ PDF  | Patient Safety     | PS-01_Patient_Identification_Policy.pdf
✅ DOCX | Patient Safety     | PS-02_Infection_Prevention_Guidelines.docx
✅ TXT  | Patient Safety     | PS-03_Fall_Prevention_Policy.txt
✅ PDF  | Pharmacy           | PH-01_Medication_Dispensing_Policy.pdf
✅ TXT  | Pharmacy           | PH-02_High_Alert_Medication_Guidelines.txt
✅ CSV  | Pharmacy           | PH-03_Pharmacy_Counter_Timings.csv
✅ DOCX | Nursing            | NR-01_Nursing_Handover_Procedure.docx
✅ PDF  | Nursing            | NR-02_Vital_Signs_Monitoring_Policy.pdf
✅ PDF  | Billing & Finance  | BF-01_Billing_Policy.pdf
✅ CSV  | Billing & Finance  | BF-02_Accepted_Payment_Methods.csv
✅ DOCX | Billing & Finance  | BF-03_Insurance

In [9]:
eval_rows = [
# (question, expected_document, expected_department, key_points, answer_available)
("What is the procedure for admitting an emergency patient?", "EM-01_Emergency_Admission_Policy.pdf", "Emergency",
 "Triage within 5 minutes; quick registration with emergency MRN; doctor assessment and consent; disposition decision; admission order and bed request to Bed Management ext 1150; ID wristband and SBAR handover; billing counselling only after stabilisation", "Yes"),
("How soon must the triage nurse assess a patient after arrival in the ED?", "EM-02_Triage_Guidelines.docx", "Emergency",
 "Within 5 minutes of arrival", "Yes"),
("Within how many minutes should a Category 3 (Urgent) patient be seen?", "EM-02_Triage_Guidelines.docx", "Emergency",
 "Category 3 is Yellow; seen within 30 minutes", "Yes"),
("Can an emergency patient be turned away if they cannot pay the admission deposit?", "EM-01_Emergency_Admission_Policy.pdf", "Emergency",
 "No; treatment is never delayed for payment; deposit can be deferred up to 24 hours after stabilisation", "Yes"),
("Which documents are needed to register a new adult patient?", "ADM-01_Patient_Registration_Procedure.docx", "Admissions",
 "Original CNIC (passport for foreigners); mobile number; insurance or panel card if applicable; referral letter if any", "Yes"),
("By what time should a planned discharge order be written and what is the checkout time?", "ADM-02_Inpatient_Admission_and_Discharge_Procedure.txt", "Admissions",
 "Discharge order by 10:00 AM; checkout time 12:00 noon", "Yes"),
("Which two identifiers must staff use to confirm a patient's identity?", "PS-01_Patient_Identification_Policy.pdf", "Patient Safety",
 "Full name plus MRN or date of birth; never room or bed number", "Yes"),
("What does a red wristband mean?", "PS-01_Patient_Identification_Policy.pdf", "Patient Safety",
 "Allergy alert", "Yes"),
("How long should alcohol-based hand rub be rubbed on the hands?", "PS-02_Infection_Prevention_Guidelines.docx", "Patient Safety",
 "20 to 30 seconds until dry; soap and water 40 to 60 seconds when hands are visibly soiled", "Yes"),
("What should I do after a needle-stick injury?", "PS-02_Infection_Prevention_Guidelines.docx", "Patient Safety",
 "Wash with soap and running water, do not squeeze; inform nurse in-charge; go to ED or Occupational Health within 2 hours; inform Infection Control ext 1300; incident report within 24 hours", "Yes"),
("At what Morse Fall Scale score is a patient considered high risk?", "PS-03_Fall_Prevention_Policy.txt", "Patient Safety",
 "Score 45 or above; yellow wristband; hourly rounding", "Yes"),
("What details must a prescription contain before the pharmacy dispenses it?", "PH-01_Medication_Dispensing_Policy.pdf", "Pharmacy",
 "Patient name and MRN; age (weight for children); generic name, strength, dose, route, frequency, duration; prescriber name, signature, PMDC number; date", "Yes"),
("How are controlled drugs stored and checked?", "PH-01_Medication_Dispensing_Policy.pdf", "Pharmacy",
 "Double-lock cabinet; counted by two persons at every shift change; discrepancies reported immediately to Pharmacy In-charge", "Yes"),
("Which medicines are classified as high-alert medications?", "PH-02_High_Alert_Medication_Guidelines.txt", "Pharmacy",
 "Insulin, heparin, concentrated potassium chloride, magnesium sulfate, opioids, chemotherapy agents, neuromuscular blocking agents; independent double-check by two nurses", "Yes"),
("What are the OPD pharmacy timings?", "PH-03_Pharmacy_Counter_Timings.csv", "Pharmacy",
 "08:00 to 22:00 daily; extension 1530", "Yes"),
("When do nursing handovers happen and which format is used?", "NR-01_Nursing_Handover_Procedure.docx", "Nursing",
 "7:00 AM, 3:00 PM and 11:00 PM; SBAR format; at the bedside; complete within 30 minutes", "Yes"),
("Which Early Warning Score requires calling the Rapid Response Team?", "NR-02_Vital_Signs_Monitoring_Policy.pdf", "Nursing",
 "Score of 7 or more; call extension 1777 immediately", "Yes"),
("How much does an ICU bed cost per day and what deposit is required?", "BF-04_Room_and_Bed_Charges.csv", "Billing & Finance",
 "PKR 35,000 per day; deposit PKR 100,000", "Yes"),
("Is there any fee for paying by credit card?", "BF-02_Accepted_Payment_Methods.csv", "Billing & Finance",
 "Yes, 1.5 percent processing fee; debit cards have no fee", "Yes"),
("How long does it take to get a refund of an unused deposit?", "BF-03_Insurance_and_Refund_Policy.docx", "Billing & Finance",
 "Cash up to PKR 20,000 immediately at counter; above PKR 20,000 by transfer or crossed cheque within 7 working days; card refunds to same card within 10 working days", "Yes"),
("What is the process for cashless insurance treatment and pre-authorization?", "BF-03_Insurance_and_Refund_Policy.docx", "Billing & Finance",
 "Panel insurers only; card and CNIC to Insurance Desk ext 1710 at least 1 working day before planned admission; insurer replies in about 4 working hours; emergencies: insurer informed within 24 hours; 10 percent co-payment", "Yes"),

# ---- Answers NOT available in the documents (hallucination tests) ----
("What is the monthly salary of a staff nurse at SHIFA?", "NONE", "NONE",
 "Should say the answer could not be found in the available documents", "No"),
("How can an outpatient book an MRI scan?", "NONE", "NONE",
 "Should say the answer could not be found in the available documents", "No"),
("What is the recommended paracetamol dose for a 5-year-old child?", "NONE", "NONE",
 "Should say the answer could not be found; must not invent a clinical dose", "No"),
("What is the hospital policy on organ donation and transplant?", "NONE", "NONE",
 "Should say the answer could not be found in the available documents", "No"),
("How much is the parking fee for visitors?", "NONE", "NONE",
 "Should say the answer could not be found in the available documents", "No"),
("Who is the current CEO of SHIFA General Hospital?", "NONE", "NONE",
 "Should say the answer could not be found in the available documents", "No"),
]

eval_df = pd.DataFrame(eval_rows, columns=["question", "expected_document", "expected_department",
                                            "expected_answer_key_points", "answer_available"])
eval_df.to_csv(PROJECT_DIR / "evaluation_questions.csv", index=False)
print("✅ evaluation_questions.csv saved")
print(eval_df["answer_available"].value_counts())

✅ evaluation_questions.csv saved
answer_available
Yes    21
No      6
Name: count, dtype: int64


In [10]:
# 1. ZIP banao
zip_path = PROJECT_DIR / "hospital_documents.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(DOCS_DIR.iterdir()):
        if f.is_file():
            z.write(f, arcname=f"documents/{f.name}")
print("✅ ZIP created:", zip_path.name, "| files inside:", len(zipfile.ZipFile(zip_path).namelist()))

# 2. Format-wise count
from collections import Counter
print("Formats:", dict(Counter(f.suffix for f in DOCS_DIR.iterdir() if f.is_file())))

# 3. Har format ka ek test
pdf = fitz.open(str(DOCS_DIR / "EM-01_Emergency_Admission_Policy.pdf"))
print("\n--- PDF test ---")
print("Pages:", len(pdf))
print(pdf[0].get_text()[:250])

d = Document(str(DOCS_DIR / "EM-02_Triage_Guidelines.docx"))
print("\n--- DOCX test ---")
print("Paragraphs:", len(d.paragraphs), "|", d.paragraphs[1].text)

print("\n--- TXT test ---")
print((DOCS_DIR / "PS-03_Fall_Prevention_Policy.txt").read_text(encoding="utf-8")[:250])

print("\n--- CSV test ---")
print(pd.read_csv(DOCS_DIR / "BF-04_Room_and_Bed_Charges.csv", comment="#").head(3))

✅ ZIP created: hospital_documents.zip | files inside: 16
Formats: {'.docx': 5, '.pdf': 5, '.txt': 3, '.csv': 3}

--- PDF test ---
Pages: 2
Emergency Admission Policy
SHIFA General Hospital (Fictional)
DEMO / SYNTHETIC HOSPITAL DATA — NOT FOR REAL CLINICAL USE
Document ID: EM-01 | Department: Emergency | Version: 3.2 | Eﬀective: 01 January 2026 | Owner: Head of
Emergency Department
1. Pu

--- DOCX test ---
Paragraphs: 30 | DEMO / SYNTHETIC HOSPITAL DATA — NOT FOR REAL CLINICAL USE

--- TXT test ---
DEMO / SYNTHETIC HOSPITAL DATA — NOT FOR REAL CLINICAL USE
SHIFA GENERAL HOSPITAL (FICTIONAL)
FALL PREVENTION POLICY

--- CSV test ---
           room_type  daily_charge_pkr  deposit_required_pkr  \
0       General Ward              4500                 20000   
1  Semi-Private Room              9000                 35000   
2       Private Room             15000                 50000   

                                      includes              notes  
0               Bed, nursing care, ba

# PHASE 3

In [11]:
%%writefile /content/drive/MyDrive/shifa-knowledge-assistant/document_loader.py
"""
document_loader.py  (Phase 3)
PDF / DOCX / TXT / CSV documents se text nikalna, saaf karna, aur metadata jodna.

Har document "records" ki list ban jaata hai. Ek record = ek page ka text + metadata:
{
    "filename":   "EM-01_Emergency_Admission_Policy.pdf",
    "doc_id":     "EM-01",
    "title":      "Emergency Admission Policy",
    "department": "Emergency",
    "file_type":  "pdf",
    "page":       1,
    "text":       "...saaf kiya hua text..."
}
"""

import re
from pathlib import Path

import fitz                      # PyMuPDF: PDF padhne ke liye
import pandas as pd              # CSV padhne ke liye
from docx import Document        # python-docx: DOCX padhne ke liye

# Filename ke prefix se department pata karte hain
DEPARTMENTS = {
    "EM": "Emergency",
    "ADM": "Admissions",
    "PS": "Patient Safety",
    "PH": "Pharmacy",
    "NR": "Nursing",
    "BF": "Billing & Finance",
}

SUPPORTED_EXTENSIONS = {".pdf", ".docx", ".txt", ".csv"}

# Disclaimer + PDF footer ("... | EM-01 | Page 1 of 2") ko pakadne wala pattern.
# Ye text search ke liye kachra hai, isliye hum ise hata dete hain.
DISCLAIMER_PATTERN = re.compile(
    r"DEMO\s*/\s*SYNTHETIC HOSPITAL DATA\s*[—–-]\s*NOT FOR REAL CLINICAL USE"
    r"(\s*\|\s*[A-Z]+-\d+\s*\|\s*Page\s*\d+\s*of\s*\d+)?",
    flags=re.IGNORECASE,
)


# ---------------------------------------------------------------
# 1) Metadata: filename se ID, title, department
# ---------------------------------------------------------------
def get_doc_info(path):
    """'EM-01_Emergency_Admission_Policy.pdf' -> ('EM-01', 'Emergency Admission Policy', 'Emergency')"""
    stem = Path(path).stem
    doc_id, _, rest = stem.partition("_")
    prefix = doc_id.split("-")[0]
    title = rest.replace("_", " ") or stem
    department = DEPARTMENTS.get(prefix, "General")
    return doc_id, title, department


# ---------------------------------------------------------------
# 2) Text cleaning
# ---------------------------------------------------------------
def clean_text(text):
    """Kachra hatao: disclaimer, footer, ====== lines, extra spaces, extra blank lines."""
    text = text.replace("\r\n", "\n").replace("\r", "\n").replace("\u00a0", " ")
    text = DISCLAIMER_PATTERN.sub("", text)

    cleaned_lines = []
    for line in text.split("\n"):
        line = re.sub(r"[ \t]+", " ", line).strip()       # extra spaces hatao
        if re.fullmatch(r"[=\-_*]{3,}", line):             # ====== ya ------ wali lines hatao
            continue
        cleaned_lines.append(line)

    text = "\n".join(cleaned_lines)
    text = re.sub(r"\n{3,}", "\n\n", text)                 # 2 se zyada blank lines nahi
    return text.strip()


# ---------------------------------------------------------------
# 3) Har file type ka loader. Sab (page_number, text) ki list dete hain
# ---------------------------------------------------------------
def load_pdf(path):
    pages = []
    with fitz.open(str(path)) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            # "blocks" = text ke tukde (lagbhag paragraphs). Har block ki line-breaks
            # ko space se badal dete hain taaki sentences beech se na tootein.
            blocks = page.get_text("blocks", sort=True)
            paragraphs = []
            for block in blocks:
                if block[6] != 0:          # 0 = text block, baaki images hain
                    continue
                paragraphs.append(block[4].replace("\n", " "))
            pages.append((page_number, clean_text("\n\n".join(paragraphs))))
    return pages


def load_docx(path):
    doc = Document(str(path))
    lines = []
    for para in doc.paragraphs:
        text = para.text.strip()
        if not text:
            continue
        if para.style.name.startswith("List"):     # bullet points ko "- " se dikhao
            text = "- " + text
        lines.append(text)

    # Agar DOCX mein tables hon to unki rows bhi add kar do
    for table in doc.tables:
        for row in table.rows:
            cells = [cell.text.strip() for cell in row.cells]
            lines.append(" | ".join(cells))

    return [(1, clean_text("\n\n".join(lines)))]   # DOCX mein asli page nahi hota


def load_txt(path):
    text = Path(path).read_text(encoding="utf-8")
    return [(1, clean_text(text))]


def load_csv(path):
    # comment="#"          -> '#' se shuru hone wali metadata lines skip
    # dtype=str            -> sab kuch text ki tarah padho
    # keep_default_na=False -> "None" jaisa text NaN (khaali) na ban jaaye
    df = pd.read_csv(path, comment="#", dtype=str, keep_default_na=False)

    rows = []
    for _, row in df.iterrows():
        parts = []
        for column, value in row.items():
            value = str(value).strip()
            if value in ("", "-"):
                continue
            parts.append(f"{column.replace('_', ' ')}: {value}")
        rows.append(" | ".join(parts))

    return [(1, clean_text("\n\n".join(rows)))]    # har row ek alag paragraph


# ---------------------------------------------------------------
# 4) Ek file load karo, aur metadata jodo
# ---------------------------------------------------------------
def load_document(path):
    path = Path(path)
    doc_id, title, department = get_doc_info(path)
    extension = path.suffix.lower()

    if extension == ".pdf":
        pages = load_pdf(path)
    elif extension == ".docx":
        pages = load_docx(path)
    elif extension == ".txt":
        pages = load_txt(path)
    elif extension == ".csv":
        pages = load_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {extension}")

    records = []
    for page_number, text in pages:
        if not text:                                # khaali page skip
            continue
        records.append({
            "filename": path.name,
            "doc_id": doc_id,
            "title": title,
            "department": department,
            "file_type": extension.lstrip("."),
            "page": page_number,
            "text": text,
        })
    return records


# ---------------------------------------------------------------
# 5) Poore folder ki saari files load karo
# ---------------------------------------------------------------
def load_all_documents(docs_dir):
    all_records = []
    for path in sorted(Path(docs_dir).iterdir()):
        if not path.is_file() or path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue
        try:
            all_records.extend(load_document(path))
        except Exception as error:                  # ek file kharab ho to baaki chalti rahein
            print(f"⚠️  {path.name} load nahi ho payi: {error}")
    return all_records


if __name__ == "__main__":
    import sys
    folder = sys.argv[1] if len(sys.argv) > 1 else "documents"
    records = load_all_documents(folder)
    print(f"Loaded {len(records)} page-records from {folder}")

Writing /content/drive/MyDrive/shifa-knowledge-assistant/document_loader.py


In [12]:
import sys, importlib
import pandas as pd

sys.path.append(str(PROJECT_DIR))
import document_loader
importlib.reload(document_loader)        # file badli ho to naya version uthao
from document_loader import load_all_documents

records = load_all_documents(PROJECT_DIR / "documents")

# --- Summary table ---
summary = (pd.DataFrame(records)
           .assign(characters=lambda d: d["text"].str.len())
           .groupby(["doc_id", "file_type", "department", "filename"], as_index=False)
           .agg(pages=("page", "count"), characters=("characters", "sum")))
print(summary.to_string(index=False))

print("\nTotal files  :", summary.shape[0])
print("Total records:", len(records))
print("Departments  :", sorted({r['department'] for r in records}))

# --- Automatic checks ---
problems = []
for r in records:
    if "SYNTHETIC HOSPITAL DATA" in r["text"].upper():
        problems.append(f"Disclaimer bacha hua hai: {r['filename']} (page {r['page']})")
    if len(r["text"]) < 100:
        problems.append(f"Text bahut chhota hai: {r['filename']} (page {r['page']})")

bf02 = " ".join(r["text"] for r in records if r["doc_id"] == "BF-02")
bf04 = " ".join(r["text"] for r in records if r["doc_id"] == "BF-04")
if "processing fee: None" not in bf02:
    problems.append("BF-02 mein 'None' value gayab hai (NaN ban gayi)")
if "daily charge pkr: 35000" not in bf04:
    problems.append("BF-04 mein ICU ka charge nahi mila")

print("\n" + ("✅ Saare checks pass!" if not problems else "❌ Problems:\n- " + "\n- ".join(problems)))

doc_id file_type        department                                               filename  pages  characters
ADM-01      docx        Admissions             ADM-01_Patient_Registration_Procedure.docx      1        2599
ADM-02       txt        Admissions ADM-02_Inpatient_Admission_and_Discharge_Procedure.txt      1        2433
 BF-01       pdf Billing & Finance                               BF-01_Billing_Policy.pdf      2        2223
 BF-02       csv Billing & Finance                     BF-02_Accepted_Payment_Methods.csv      1        1729
 BF-03      docx Billing & Finance                 BF-03_Insurance_and_Refund_Policy.docx      1        2504
 BF-04       csv Billing & Finance                         BF-04_Room_and_Bed_Charges.csv      1        1655
 EM-01       pdf         Emergency                   EM-01_Emergency_Admission_Policy.pdf      2        3927
 EM-02      docx         Emergency                           EM-02_Triage_Guidelines.docx      1        2862
 NR-01      docx   

In [13]:
def show(doc_id, page=1, n=800):
    r = next(r for r in records if r["doc_id"] == doc_id and r["page"] == page)
    print(f"===== {r['filename']} | {r['department']} | page {r['page']} =====")
    print(r["text"][:n])
    print("\n")

show("EM-01")          # PDF
show("EM-02")          # DOCX
show("PS-03")          # TXT
show("BF-04", n=600)   # CSV

===== EM-01_Emergency_Admission_Policy.pdf | Emergency | page 1 =====
Emergency Admission Policy

SHIFA General Hospital (Fictional)

Document ID: EM-01 | Department: Emergency | Version: 3.2 | Eﬀective: 01 January 2026 | Owner: Head of Emergency Department

1. Purpose and Scope

This policy explains how patients who arrive at the SHIFA General Hospital Emergency Department (ED) are received, registered, assessed and, when needed, admitted. It applies to all ED doctors, nurses, registration clerks, security staﬀ and the Bed Management team.

The ED is open 24 hours a day, 7 days a week. The ED direct extension is 1000.

2. Guiding Principles

• Assessment and stabilisation come ﬁrst. Paperwork, identity documents, deposits and insurance approvals must never delay life-saving treatment. • No patient is refused initial assessment because of inability to pay, a


===== EM-02_Triage_Guidelines.docx | Emergency | page 1 =====
Triage Guidelines

Document ID: EM-02 | Department: Emergency | V

# Phase 4

In [14]:
%%writefile /content/drive/MyDrive/shifa-knowledge-assistant/chunker.py
"""
chunker.py  (Phase 4)
Document_loader se aaye "page records" ko chhote, overlapping chunks mein todta hai,
aur har chunk ke saath metadata + unique chunk_id jodta hai.
"""

import re

CHUNK_SIZE = 800     # har chunk mein lagbhag itne characters
CHUNK_OVERLAP = 150  # consecutive chunks ke beech itne characters repeat honge


# ---------------------------------------------------------------
# 1) Text ko paragraphs aur sentences mein todne ke helpers
# ---------------------------------------------------------------
def split_into_paragraphs(text):
    return [p.strip() for p in text.split("\n\n") if p.strip()]


def split_into_sentences(text):
    # Simple sentence splitter: '.', '!', '?' ke baad space + capital letter/number par todo.
    # Hospital policy text ke liye ye kaafi achha kaam karta hai.
    sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", text)
    return [s.strip() for s in sentences if s.strip()]


# ---------------------------------------------------------------
# 2) Paragraphs ki list ko CHUNK_SIZE ke hisaab se jodo/todo
# ---------------------------------------------------------------
def pack_paragraphs(paragraphs, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP,
                     split_long_by_sentence=True):
    chunks = []
    current = ""

    for para in paragraphs:
        # Agar ek paragraph akela hi chunk_size se bada hai...
        if len(para) > chunk_size:
            if current:
                chunks.append(current.strip())
                current = ""
            if split_long_by_sentence:
                # ...to use sentences mein tod kar normal tarike se pack karo
                sentences = split_into_sentences(para)
                for sent in sentences:
                    if current and len(current) + len(sent) + 1 > chunk_size:
                        chunks.append(current.strip())
                        current = current[-overlap:] + " " + sent
                    else:
                        current = (current + " " + sent).strip()
            else:
                # CSV rows: pura row ek chunk mein hi rehne do (todo mat)
                chunks.append(para.strip())
            continue

        # Normal paragraph: current chunk mein fit hota hai to jodo, warna naya chunk shuru karo
        if current and len(current) + len(para) + 2 > chunk_size:
            chunks.append(current.strip())
            current = current[-overlap:] + "\n\n" + para
        else:
            current = (current + "\n\n" + para).strip() if current else para

    if current.strip():
        chunks.append(current.strip())

    return [c for c in chunks if c]


# ---------------------------------------------------------------
# 3) Ek page-record ko chunk-records mein badlo
# ---------------------------------------------------------------
def chunk_record(record, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    paragraphs = split_into_paragraphs(record["text"])

    # CSV ke liye: har row apne aap mein poora rehna chahiye, isliye sentence-split band
    split_long = record["file_type"] != "csv"

    pieces = pack_paragraphs(paragraphs, chunk_size, overlap, split_long_by_sentence=split_long)

    chunks = []
    for i, piece in enumerate(pieces):
        chunks.append({
            "chunk_id": f"{record['doc_id']}_p{record['page']}_c{i}",
            "filename": record["filename"],
            "doc_id": record["doc_id"],
            "title": record["title"],
            "department": record["department"],
            "file_type": record["file_type"],
            "page": record["page"],
            "chunk_index_in_page": i,
            "text": piece,
        })
    return chunks


# ---------------------------------------------------------------
# 4) Saare records ek saath chunk karo
# ---------------------------------------------------------------
def chunk_all_records(records, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    all_chunks = []
    for record in records:
        all_chunks.extend(chunk_record(record, chunk_size, overlap))
    return all_chunks


if __name__ == "__main__":
    import sys
    from pathlib import Path
    sys.path.append(str(Path(__file__).parent))
    from document_loader import load_all_documents

    recs = load_all_documents("documents")
    chks = chunk_all_records(recs)
    print(f"{len(recs)} page-records -> {len(chks)} chunks")

Writing /content/drive/MyDrive/shifa-knowledge-assistant/chunker.py


In [15]:
import importlib
import pandas as pd

import chunker
importlib.reload(chunker)
from chunker import chunk_all_records

chunks = chunk_all_records(records)

df = pd.DataFrame(chunks)
df["char_len"] = df["text"].str.len()

print("Total chunks:", len(chunks))
print("\nChunks per document:")
print(df.groupby(["doc_id", "filename"], as_index=False)
        .agg(chunks=("chunk_id", "count"), avg_chars=("char_len", "mean"))
        .to_string(index=False))

print("\nChunk length stats (characters):")
print(df["char_len"].describe()[["min", "mean", "max"]])

Total chunks: 67

Chunks per document:
doc_id                                               filename  chunks  avg_chars
ADM-01             ADM-01_Patient_Registration_Procedure.docx       5 639.800000
ADM-02 ADM-02_Inpatient_Admission_and_Discharge_Procedure.txt       4 720.500000
 BF-01                               BF-01_Billing_Policy.pdf       5 534.600000
 BF-02                     BF-02_Accepted_Payment_Methods.csv       3 676.333333
 BF-03                 BF-03_Insurance_and_Refund_Policy.docx       4 738.500000
 BF-04                         BF-04_Room_and_Bed_Charges.csv       3 651.333333
 EM-01                   EM-01_Emergency_Admission_Policy.pdf       7 667.857143
 EM-02                           EM-02_Triage_Guidelines.docx       5 692.000000
 NR-01                  NR-01_Nursing_Handover_Procedure.docx       3 740.000000
 NR-02                NR-02_Vital_Signs_Monitoring_Policy.pdf       4 636.500000
 PH-01                 PH-01_Medication_Dispensing_Policy.pdf       5 

In [16]:
problems = []

# Check 1: chunk_id hamesha unique hona chahiye
ids = [c["chunk_id"] for c in chunks]
if len(ids) != len(set(ids)):
    problems.append("Duplicate chunk_id mila!")

# Check 2: koi chunk bahut chhota (noise) ya bahut bada (chunking fail) nahi hona chahiye
for c in chunks:
    if len(c["text"]) < 30:
        problems.append(f"Bahut chhota chunk: {c['chunk_id']} ({len(c['text'])} chars)")
    if len(c["text"]) > 1200:
        problems.append(f"Bahut bada chunk: {c['chunk_id']} ({len(c['text'])} chars)")

# Check 3: CSV ki har row apne-aap mein poori honi chahiye (beech se na kate)
csv_chunks = [c for c in chunks if c["file_type"] == "csv"]
for c in csv_chunks:
    if "|" not in c["text"]:
        problems.append(f"CSV chunk mein '|' nahi mila, row toot gayi ho sakti hai: {c['chunk_id']}")

# Check 4: overlap kaam kar raha hai ya nahi — kisi multi-chunk document mein
em01_chunks = [c for c in chunks if c["doc_id"] == "EM-01" and c["page"] == 1]
if len(em01_chunks) >= 2:
    end_of_first = em01_chunks[0]["text"][-50:]
    start_of_second = em01_chunks[1]["text"][:200]
    overlap_found = any(word in start_of_second for word in end_of_first.split() if len(word) > 4)
    if not overlap_found:
        problems.append("EM-01 ke chunks ke beech overlap nahi mila (check zaroor karo, false alarm bhi ho sakta hai)")

print("✅ Saare checks pass!" if not problems else "❌ Problems:\n- " + "\n- ".join(problems))

✅ Saare checks pass!


In [18]:
def show_chunks(doc_id):
    doc_chunks = [c for c in chunks if c["doc_id"] == doc_id]
    print(f"===== {doc_id} | total {len(doc_chunks)} chunks =====\n")
    for c in doc_chunks:
        print(f"--- {c['chunk_id']} (page {c['page']}, {len(c['text'])} chars) ---")
        print(c["text"])
        print()

show_chunks("PH-01")

===== PH-01 | total 5 chunks =====

--- PH-01_p1_c0 (page 1, 730 chars) ---
Medication Dispensing Policy

SHIFA General Hospital (Fictional)

Document ID: PH-01 | Department: Pharmacy | Version: 4.0 | Eﬀective: 01 April 2026 | Owner: Chief Pharmacist

1. Purpose

This policy ensures that every medicine dispensed at SHIFA General Hospital is correct, safe and properly labelled. It applies to all pharmacists and pharmacy technicians. The Pharmacy main extension is 1500.

2. Valid Prescription

A medicine is dispensed only against a valid prescription or medication order that contains:

• Patient name and MRN. • Age, and body weight for children. • Generic drug name, strength, dose, route, frequency and duration. • Prescriber name, signature and PMDC registration number. • Date of the prescription.

--- PH-01_p1_c1 (page 1, 784 chars) ---
neric drug name, strength, dose, route, frequency and duration. • Prescriber name, signature and PMDC registration number. • Date of the prescription.



# PHASE#6

In [19]:
from document_loader import load_all_documents
from chunker import chunk_all_records

records = load_all_documents(PROJECT_DIR / "documents")
chunks = chunk_all_records(records)
print("Records:", len(records), "| Chunks:", len(chunks))

Records: 20 | Chunks: 67


In [20]:
%%writefile /content/drive/MyDrive/shifa-knowledge-assistant/embedder.py
"""
embedder.py  (Phase 5)
Chunk ke text ko embeddings (numbers ki list) mein badalta hai, sentence-transformers
ka 'all-MiniLM-L6-v2' model use karke. Ye wahi model hai jo build_index.py (Phase 6)
aur app.py (Phase 10) dono mein use hoga, taaki numbers compare-able rahein.
"""

from sentence_transformers import SentenceTransformer

MODEL_NAME = "all-MiniLM-L6-v2"

_model = None   # model ek baar load hoke cache mein reh jaata hai (bar bar load na ho)


def get_model():
    global _model
    if _model is None:
        print(f"Loading embedding model: {MODEL_NAME} (pehli baar thoda time lagega)...")
        _model = SentenceTransformer(MODEL_NAME)
    return _model


def embed_texts(texts, batch_size=32, show_progress=True):
    """
    Texts ki list leta hai, embeddings (numbers ki list ki list) wapas deta hai.
    Example: embed_texts(["hello", "world"]) -> numpy array shape (2, 384)
    """
    model = get_model()
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=show_progress,
        normalize_embeddings=True,   # FAISS similarity search ke liye zaroori (Phase 6 mein)
    )
    return embeddings


def embed_chunks(chunks, batch_size=32):
    """Chunk-records ki list leta hai, har chunk ka 'text' embed karta hai."""
    texts = [c["text"] for c in chunks]
    return embed_texts(texts, batch_size=batch_size)


if __name__ == "__main__":
    sample = ["What is the triage process?", "ICU room charges per day"]
    vectors = embed_texts(sample)
    print("Shape:", vectors.shape)

Writing /content/drive/MyDrive/shifa-knowledge-assistant/embedder.py


In [21]:
import importlib
import embedder
importlib.reload(embedder)
from embedder import embed_chunks

embeddings = embed_chunks(chunks)

print("Embeddings shape:", embeddings.shape)
print("Chunks count    :", len(chunks))
print("Dimension       :", embeddings.shape[1], "(all-MiniLM-L6-v2 hamesha 384 deta hai)")

Loading embedding model: all-MiniLM-L6-v2 (pehli baar thoda time lagega)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embeddings shape: (67, 384)
Chunks count    : 67
Dimension       : 384 (all-MiniLM-L6-v2 hamesha 384 deta hai)


In [22]:
import numpy as np
from embedder import embed_texts

test_sentences = [
    "What is the triage process in the emergency department?",   # 0: Emergency-related
    "How are patients sorted by urgency when they arrive?",       # 1: same meaning as 0, alag words
    "What is the ICU room charge per day?",                       # 2: Billing-related, bilkul alag topic
]

vecs = embed_texts(test_sentences, show_progress=False)

# normalize_embeddings=True hone ki wajah se, dot product hi cosine similarity hai
sim_0_1 = np.dot(vecs[0], vecs[1])   # dono emergency/triage ke baare mein -> high honi chahiye
sim_0_2 = np.dot(vecs[0], vecs[2])   # triage vs billing -> low honi chahiye

print(f"Similarity (triage vs sorting-by-urgency) : {sim_0_1:.3f}   <- high honi chahiye")
print(f"Similarity (triage vs ICU billing)         : {sim_0_2:.3f}   <- low honi chahiye")
print("\n✅ Model meaning samajh raha hai!" if sim_0_1 > sim_0_2 else "❌ Kuch gadbad hai")

Similarity (triage vs sorting-by-urgency) : 0.488   <- high honi chahiye
Similarity (triage vs ICU billing)         : 0.321   <- low honi chahiye

✅ Model meaning samajh raha hai!


In [23]:
import pickle

KB_DIR = PROJECT_DIR / "knowledge_base"
KB_DIR.mkdir(exist_ok=True)

# Har chunk record mein uski embedding bhi daal do (taaki dono saath rahein)
for chunk, vector in zip(chunks, embeddings):
    chunk["embedding"] = vector

with open(KB_DIR / "chunks_with_embeddings.pkl", "wb") as f:
    pickle.dump(chunks, f)

print("✅ Saved:", KB_DIR / "chunks_with_embeddings.pkl")
print("File size:", round((KB_DIR / "chunks_with_embeddings.pkl").stat().st_size / 1024, 1), "KB")

# Verify: wapas load karke check karo
with open(KB_DIR / "chunks_with_embeddings.pkl", "rb") as f:
    reloaded = pickle.load(f)

print("Reloaded chunks:", len(reloaded))
print("Pehle chunk ka embedding shape:", reloaded[0]["embedding"].shape)
print("Pehle chunk ka text (pehle 100 chars):", reloaded[0]["text"][:100])

✅ Saved: /content/drive/MyDrive/shifa-knowledge-assistant/knowledge_base/chunks_with_embeddings.pkl
File size: 151.0 KB
Reloaded chunks: 67
Pehle chunk ka embedding shape: (384,)
Pehle chunk ka text (pehle 100 chars): Patient Registration Procedure

Document ID: ADM-01 | Department: Admissions | Version: 2.1 | Effect


# Phase 6

In [24]:
import sys
from pathlib import Path
PROJECT_DIR = Path("/content/drive/MyDrive/shifa-knowledge-assistant")
sys.path.append(str(PROJECT_DIR))

import pickle

KB_DIR = PROJECT_DIR / "knowledge_base"

with open(KB_DIR / "chunks_with_embeddings.pkl", "rb") as f:
    chunks = pickle.load(f)

print("Loaded chunks:", len(chunks))
print("Embedding shape of chunk 0:", chunks[0]["embedding"].shape)

Loaded chunks: 67
Embedding shape of chunk 0: (384,)


In [25]:
%%writefile /content/drive/MyDrive/shifa-knowledge-assistant/build_index.py
"""
build_index.py  (Phase 6)
Hospital documents se poora knowledge base banata hai aur SAVE karta hai:
  knowledge_base/faiss.index   -> FAISS search structure
  knowledge_base/chunks.pkl    -> har chunk ka text + metadata (FAISS position se match)
  knowledge_base/metadata.pkl  -> build info (kab bana, kitne chunks, konsa model)

Isko sirf ek baar chalana hai (ya jab documents badlein). Streamlit app (app.py) ye
saved files sirf LOAD karega, documents ko dobara process nahi karega.
"""

import pickle
from datetime import datetime
from pathlib import Path

import faiss
import numpy as np

from document_loader import load_all_documents
from chunker import chunk_all_records
from embedder import embed_chunks, MODEL_NAME

DOCS_DIR = Path("documents")
KB_DIR = Path("knowledge_base")


def build_and_save_index(docs_dir=DOCS_DIR, kb_dir=KB_DIR):
    kb_dir = Path(kb_dir)
    kb_dir.mkdir(parents=True, exist_ok=True)

    # 1. Documents load + clean karo
    print("Step 1/4: Documents load ho rahi hain...")
    records = load_all_documents(docs_dir)
    print(f"   {len(records)} page-records mile")

    # 2. Chunks banao
    print("Step 2/4: Chunking ho rahi hai...")
    chunks = chunk_all_records(records)
    print(f"   {len(chunks)} chunks bane")

    # 3. Embeddings banao
    print("Step 3/4: Embeddings ban rahi hain (ek baar ka kaam)...")
    embeddings = embed_chunks(chunks)
    embeddings = np.array(embeddings).astype("float32")   # FAISS ko float32 chahiye
    print(f"   Embeddings shape: {embeddings.shape}")

    # 4. FAISS index banao
    #    IndexFlatIP = Inner Product search. Kyunki embeddings normalized hain
    #    (Phase 5 mein normalize_embeddings=True), Inner Product == Cosine Similarity.
    #    "Flat" matlab exact search (approximate nahi) - itne kam chunks ke liye perfect hai.
    print("Step 4/4: FAISS index ban raha hai...")
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)
    print(f"   Index mein {index.ntotal} vectors hain")

    # chunk se "embedding" key hata do (wo ab FAISS mein hai, dobara save karne ki zaroorat nahi)
    chunks_to_save = []
    for c in chunks:
        c = dict(c)
        c.pop("embedding", None)
        chunks_to_save.append(c)

    # 5. Save: FAISS index
    faiss.write_index(index, str(kb_dir / "faiss.index"))

    # 6. Save: chunks (text + metadata, FAISS position ke order mein)
    with open(kb_dir / "chunks.pkl", "wb") as f:
        pickle.dump(chunks_to_save, f)

    # 7. Save: metadata (debugging/reference ke liye)
    meta = {
        "built_at": datetime.now().isoformat(),
        "num_documents": len({c["doc_id"] for c in chunks_to_save}),
        "num_chunks": len(chunks_to_save),
        "embedding_model": MODEL_NAME,
        "embedding_dimension": dimension,
        "index_type": "IndexFlatIP",
    }
    with open(kb_dir / "metadata.pkl", "wb") as f:
        pickle.dump(meta, f)

    print("\n✅ Knowledge base ban gaya aur save ho gaya:")
    for key, value in meta.items():
        print(f"   {key}: {value}")

    return index, chunks_to_save, meta


if __name__ == "__main__":
    build_and_save_index()

Overwriting /content/drive/MyDrive/shifa-knowledge-assistant/build_index.py


In [26]:
import os
os.chdir(PROJECT_DIR)   # taaki "documents/" aur "knowledge_base/" relative paths sahi se mile

import importlib
import build_index
importlib.reload(build_index)
from build_index import build_and_save_index

index, saved_chunks, meta = build_and_save_index()

Step 1/4: Documents load ho rahi hain...
   20 page-records mile
Step 2/4: Chunking ho rahi hai...
   67 chunks bane
Step 3/4: Embeddings ban rahi hain (ek baar ka kaam)...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

   Embeddings shape: (67, 384)
Step 4/4: FAISS index ban raha hai...
   Index mein 67 vectors hain

✅ Knowledge base ban gaya aur save ho gaya:
   built_at: 2026-10-07T06:39:06.327400
   num_documents: 16
   num_chunks: 67
   embedding_model: all-MiniLM-L6-v2
   embedding_dimension: 384
   index_type: IndexFlatIP


In [28]:
import faiss
import pickle
from pathlib import Path
from embedder import embed_texts

KB_DIR = PROJECT_DIR / "knowledge_base"

# 1. Load FAISS index and chunks
loaded_index = faiss.read_index(str(KB_DIR / "faiss.index"))
with open(KB_DIR / "chunks.pkl", "rb") as f:
    loaded_chunks = pickle.load(f)

# 2. Search
query = "What is the deposit for ICU admission?"
query_vector = embed_texts([query], show_progress=False).astype("float32")

# Top 3 sabse milte-julte chunks dhundo
scores, indices = loaded_index.search(query_vector, k=3)

print(f"Query: {query}\n")
for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
    chunk = loaded_chunks[idx]
    print(f"#{rank} | score={score:.3f} | {chunk['filename']} (page {chunk['page']})")
    print(f"     {chunk['text'][:150]}...")
    print()

Query: What is the deposit for ICU admission?

#1 | score=0.627 | BF-01_Billing_Policy.pdf (page 1)
     is collected at admission:

• General Ward: PKR 20,000. • Semi-Private Room: PKR 35,000. • Private Room: PKR 50,000. • ICU, CCU and NICU: PKR 100,000....

#2 | score=0.570 | BF-04_Room_and_Bed_Charges.csv (page 1)
     ily charge pkr: 35000 | deposit required pkr: 100000 | includes: Bed, cardiac monitoring, intensive nursing care | notes: Procedures billed separately...

#3 | score=0.521 | BF-04_Room_and_Bed_Charges.csv (page 1)
     | daily charge pkr: 25000 | deposit required pkr: 75000 | includes: Bed, nursing care, meals, TV, attendant bed, sitting area | notes: Single patient
...



In [29]:
import sys
from pathlib import Path
PROJECT_DIR = Path("/content/drive/MyDrive/shifa-knowledge-assistant")
sys.path.append(str(PROJECT_DIR))

import os
os.chdir(PROJECT_DIR)

import pickle, faiss, pandas as pd
from embedder import embed_texts

KB_DIR = PROJECT_DIR / "knowledge_base"

index = faiss.read_index(str(KB_DIR / "faiss.index"))
with open(KB_DIR / "chunks.pkl", "rb") as f:
    chunks = pickle.load(f)

print("FAISS vectors:", index.ntotal, "| Chunks:", len(chunks))

FAISS vectors: 67 | Chunks: 67


In [30]:
%%writefile /content/drive/MyDrive/shifa-knowledge-assistant/retriever.py
"""
retriever.py  (Phase 7)
Saved FAISS index aur chunks load karta hai, aur ek sawaal ke liye Top-K sabse
milte-julte chunks dhoondhta hai. Isi function ko Phase 8 (RAG pipeline) aur
Phase 10 (Streamlit app) dono use karenge.
"""

import pickle
from pathlib import Path

import faiss
import numpy as np

from embedder import embed_texts

KB_DIR = Path("knowledge_base")


def load_knowledge_base(kb_dir=KB_DIR):
    """Saved FAISS index aur chunks (text+metadata) load karta hai. Documents re-process NAHI hote."""
    kb_dir = Path(kb_dir)
    index = faiss.read_index(str(kb_dir / "faiss.index"))
    with open(kb_dir / "chunks.pkl", "rb") as f:
        chunks = pickle.load(f)
    return index, chunks


def retrieve(query, index, chunks, top_k=3, min_score=0.0):
    """
    Ek sawaal ke liye Top-K sabse milte-julte chunks dhoondhta hai.
    Har result mein chunk ka text, metadata, aur uska similarity score (0 to 1) hota hai.
    """
    query_vector = embed_texts([query], show_progress=False).astype("float32")
    scores, indices = index.search(query_vector, top_k)

    results = []
    for idx, score in zip(indices[0], scores[0]):
        if idx == -1:                 # FAISS itne results na mile to -1 deta hai
            continue
        if score < min_score:
            continue
        chunk = dict(chunks[idx])
        chunk["score"] = float(score)
        results.append(chunk)

    return results


if __name__ == "__main__":
    index, chunks = load_knowledge_base()
    results = retrieve("What is the deposit for ICU admission?", index, chunks, top_k=3)
    for r in results:
        print(f"{r['score']:.3f} | {r['filename']} (p{r['page']}) | {r['text'][:100]}")


Writing /content/drive/MyDrive/shifa-knowledge-assistant/retriever.py


In [33]:
import importlib
import retriever
importlib.reload(retriever)
from retriever import load_knowledge_base, retrieve

index, chunks = load_knowledge_base()
results = retrieve("What is the deposit for ICU admission?", index, chunks, top_k=3)

for r in results:
    print(f"{r['score']:.3f} | {r['filename']} (page {r['page']})")
    print(f"      {r['text'][:120]}...\n")

0.627 | BF-01_Billing_Policy.pdf (page 1)
      is collected at admission:

• General Ward: PKR 20,000. • Semi-Private Room: PKR 35,000. • Private Room: PKR 50,000. • I...

0.570 | BF-04_Room_and_Bed_Charges.csv (page 1)
      ily charge pkr: 35000 | deposit required pkr: 100000 | includes: Bed, cardiac monitoring, intensive nursing care | notes...

0.521 | BF-04_Room_and_Bed_Charges.csv (page 1)
      | daily charge pkr: 25000 | deposit required pkr: 75000 | includes: Bed, nursing care, meals, TV, attendant bed, sitting...



In [34]:
eval_df = pd.read_csv(PROJECT_DIR / "evaluation_questions.csv")
answerable = eval_df[eval_df["answer_available"] == "Yes"].reset_index(drop=True)
print("Testable (answerable) questions:", len(answerable))

TOP_K = 3
rows = []

for _, row in answerable.iterrows():
    results = retrieve(row["question"], index, chunks, top_k=TOP_K)
    retrieved_files = [r["filename"] for r in results]

    # Hit check: kya expected_document top-K retrieved files mein hai?
    hit = row["expected_document"] in retrieved_files

    rows.append({
        "question": row["question"],
        "expected_document": row["expected_document"],
        "top1_file": results[0]["filename"] if results else None,
        "top1_score": round(results[0]["score"], 3) if results else None,
        "hit_at_3": hit,
        "retrieved_files": retrieved_files,
    })

results_df = pd.DataFrame(rows)

hit_rate = results_df["hit_at_3"].mean() * 100
print(f"\n{'='*60}")
print(f"Hit@{TOP_K} Rate: {hit_rate:.1f}%  ({results_df['hit_at_3'].sum()}/{len(results_df)} questions)")
print(f"Average top-1 score: {results_df['top1_score'].mean():.3f}")
print(f"{'='*60}\n")

# Pura table dikhao (expected vs actual)
pd.set_option('display.max_colwidth', 50)
print(results_df[["question", "expected_document", "top1_file", "top1_score", "hit_at_3"]].to_string(index=False))

Testable (answerable) questions: 21

Hit@3 Rate: 100.0%  (21/21 questions)
Average top-1 score: 0.591

                                                                               question                                      expected_document                                              top1_file  top1_score  hit_at_3
                              What is the procedure for admitting an emergency patient?                   EM-01_Emergency_Admission_Policy.pdf                   EM-01_Emergency_Admission_Policy.pdf       0.661      True
               How soon must the triage nurse assess a patient after arrival in the ED?                           EM-02_Triage_Guidelines.docx                   EM-01_Emergency_Admission_Policy.pdf       0.651      True
                  Within how many minutes should a Category 3 (Urgent) patient be seen?                           EM-02_Triage_Guidelines.docx                           EM-02_Triage_Guidelines.docx       0.779      True
      Can an emer

In [35]:
failures = results_df[~results_df["hit_at_3"]]

if len(failures) == 0:
    print("🎉 Koi failure nahi! Saari questions ne sahi document dhoonda.")
else:
    print(f"❌ {len(failures)} questions fail hui:\n")
    for _, row in failures.iterrows():
        print(f"Q: {row['question']}")
        print(f"   Expected : {row['expected_document']}")
        print(f"   Got top1 : {row['top1_file']} (score {row['top1_score']})")
        print(f"   All retrieved: {row['retrieved_files']}")
        print()

🎉 Koi failure nahi! Saari questions ne sahi document dhoonda.


In [36]:
not_answerable = eval_df[eval_df["answer_available"] == "No"]

for _, row in not_answerable.iterrows():
    results = retrieve(row["question"], index, chunks, top_k=3)
    top_score = results[0]["score"] if results else 0
    print(f"Q: {row['question']}")
    print(f"   Top1 score: {top_score:.3f} | file: {results[0]['filename'] if results else 'none'}")
    print()

Q: What is the monthly salary of a staff nurse at SHIFA?
   Top1 score: 0.463 | file: BF-01_Billing_Policy.pdf

Q: How can an outpatient book an MRI scan?
   Top1 score: 0.451 | file: ADM-01_Patient_Registration_Procedure.docx

Q: What is the recommended paracetamol dose for a 5-year-old child?
   Top1 score: 0.465 | file: NR-02_Vital_Signs_Monitoring_Policy.pdf

Q: What is the hospital policy on organ donation and transplant?
   Top1 score: 0.424 | file: EM-01_Emergency_Admission_Policy.pdf

Q: How much is the parking fee for visitors?
   Top1 score: 0.306 | file: BF-01_Billing_Policy.pdf

Q: Who is the current CEO of SHIFA General Hospital?
   Top1 score: 0.571 | file: ADM-02_Inpatient_Admission_and_Discharge_Procedure.txt



# Phase 8

In [38]:
from google.colab import userdata

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
print("Key loaded:", "✅ Yes, length =", len(GROQ_API_KEY)) if GROQ_API_KEY else print("❌ Key nahi mili")

Key loaded: ✅ Yes, length = 56


In [39]:
%%writefile /content/drive/MyDrive/shifa-knowledge-assistant/rag_pipeline.py
"""
rag_pipeline.py  (Phase 8)
Retrieved chunks ko context mein badalta hai, strict grounding instruction ke saath
Groq LLM ko bhejta hai, aur jawaab wapas deta hai. Sources (Phase 9) is file ke
upar build honge.
"""

import os
from groq import Groq

from retriever import load_knowledge_base, retrieve

GROQ_MODEL = "openai/gpt-oss-20b"

SYSTEM_PROMPT = """You are SHIFA Knowledge Assistant, a policy and procedure assistant for hospital staff at SHIFA General Hospital.

STRICT RULES:
1. Answer only from the provided hospital document context below. Do not use any outside knowledge.
2. If sufficient information is not available in the context, clearly say that the answer could not be found in the available documents. Do not guess or make up an answer.
3. When you use information from a source, refer to it using its tag, like [S1] or [S2].
4. This is a knowledge/policy assistant, not a medical diagnosis system. Do not give clinical advice beyond what is written in the policies.
5. Be concise and direct. Use short paragraphs or bullet points where helpful.
"""


def get_groq_client(api_key=None):
    api_key = api_key or os.environ.get("GROQ_API_KEY")
    if not api_key:
        raise ValueError("GROQ_API_KEY nahi mili. Isse environment variable ya argument se pass karo.")
    return Groq(api_key=api_key)


def build_context(chunks):
    """
    Retrieved chunks ko numbered source blocks mein badalta hai, jaise:
    [S1] Source: EM-01_Emergency_Admission_Policy.pdf (Emergency, page 1)
    <chunk text>
    """
    blocks = []
    for i, chunk in enumerate(chunks, start=1):
        tag = f"S{i}"
        header = f"[{tag}] Source: {chunk['filename']} ({chunk['department']}, page {chunk['page']})"
        blocks.append(f"{header}\n{chunk['text']}")
    return "\n\n---\n\n".join(blocks), blocks


def answer_question(question, index, chunks_db, client, top_k=4, temperature=0.1):
    """
    Poora RAG flow: question -> retrieve -> Groq -> grounded answer.
    Return karta hai: {"answer": str, "sources": list of retrieved chunks, "context_used": str}
    """
    # 1. Retrieve
    retrieved = retrieve(question, index, chunks_db, top_k=top_k)

    if not retrieved:
        return {
            "answer": "The answer could not be found in the available documents.",
            "sources": [],
            "context_used": "",
        }

    # 2. Context banao
    context_text, _ = build_context(retrieved)

    # 3. Groq ko prompt bhejo
    user_message = f"""Context from hospital documents:

{context_text}

---

Staff question: {question}

Answer the question using only the context above. Cite sources using their tags like [S1], [S2]."""

    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
        temperature=temperature,
        max_tokens=600,
    )

    answer_text = response.choices[0].message.content

    return {
        "answer": answer_text,
        "sources": retrieved,
        "context_used": context_text,
    }


if __name__ == "__main__":
    index, chunks_db = load_knowledge_base()
    client = get_groq_client()
    result = answer_question("What is the deposit for ICU admission?", index, chunks_db, client)
    print(result["answer"])

Overwriting /content/drive/MyDrive/shifa-knowledge-assistant/rag_pipeline.py


In [40]:
import importlib
import rag_pipeline
importlib.reload(rag_pipeline)
from rag_pipeline import get_groq_client, answer_question
from retriever import load_knowledge_base

index, chunks_db = load_knowledge_base()
client = get_groq_client(GROQ_API_KEY)

result = answer_question("What is the deposit for ICU admission?", index, chunks_db, client)

print("ANSWER:")
print(result["answer"])
print(f"\n(Used {len(result['sources'])} sources)")

ANSWER:
The deposit for an ICU admission is **PKR 100,000**【S1】【S4】.

(Used 4 sources)


In [41]:
test_questions = [
    "What is the monthly salary of a staff nurse at SHIFA?",
    "What is the recommended paracetamol dose for a 5-year-old child?",
    "Who is the current CEO of SHIFA General Hospital?",
]

for q in test_questions:
    result = answer_question(q, index, chunks_db, client)
    print(f"Q: {q}")
    print(f"A: {result['answer']}")
    print()

Q: What is the monthly salary of a staff nurse at SHIFA?
A: I’m sorry, but the available documents do not contain information about the monthly salary of a staff nurse at SHIFA General Hospital.

Q: What is the recommended paracetamol dose for a 5-year-old child?
A: I’m sorry, but the recommended paracetamol dose for a 5‑year‑old child is not provided in the documents you supplied.

Q: Who is the current CEO of SHIFA General Hospital?
A: I’m sorry, but the current CEO of SHIFA General Hospital is not mentioned in the documents provided.



In [42]:
import pandas as pd

eval_df = pd.read_csv(PROJECT_DIR / "evaluation_questions.csv")

results_log = []
for _, row in eval_df.iterrows():
    result = answer_question(row["question"], index, chunks_db, client)
    results_log.append({
        "question": row["question"],
        "answer_available_expected": row["answer_available"],
        "answer": result["answer"],
        "num_sources": len(result["sources"]),
    })
    print(f"[{row['answer_available']}] Q: {row['question'][:60]}...")
    print(f"      A: {result['answer'][:150]}...\n")

eval_results_df = pd.DataFrame(results_log)

[Yes] Q: What is the procedure for admitting an emergency patient?...
      A: **Procedure for admitting an emergency patient**

1. **Arrival & triage** – Patient is seen by a triage nurse within 5 minutes and assigned a triage c...

[Yes] Q: How soon must the triage nurse assess a patient after arriva...
      A: The triage nurse must begin the assessment **within 5 minutes of the patient’s arrival** in the ED.  
[ S2, S3 ]...

[Yes] Q: Within how many minutes should a Category 3 (Urgent) patient...
      A: A Category 3 (Urgent) patient should be seen within **30 minutes** of arrival. [S1] [S2]...

[Yes] Q: Can an emergency patient be turned away if they cannot pay t...
      A: No.  An emergency patient is **not** denied stabilising treatment because of an inability to pay the admission deposit.  The deposit may be deferred f...

[Yes] Q: Which documents are needed to register a new adult patient?...
      A: **Documents required for registering a new adult patient**

- Original CNI

In [43]:
NOT_FOUND_PHRASES = ["could not be found", "not available in the", "does not contain", "no information", "not mentioned in"]

def says_not_found(answer):
    return any(phrase in answer.lower() for phrase in NOT_FOUND_PHRASES)

eval_results_df["said_not_found"] = eval_results_df["answer"].apply(says_not_found)

# Check 1: "No" expected questions ke liye, kya LLM ne sahi se "not found" bola?
no_rows = eval_results_df[eval_results_df["answer_available_expected"] == "No"]
correct_refusals = no_rows["said_not_found"].sum()
print(f"Hallucination check: {correct_refusals}/{len(no_rows)} 'not available' questions mein LLM ne sahi refuse kiya")

# Check 2: "Yes" expected questions ke liye, kya LLM ne galti se "not found" bol diya (ye bura hai)?
yes_rows = eval_results_df[eval_results_df["answer_available_expected"] == "Yes"]
wrongly_refused = yes_rows["said_not_found"].sum()
print(f"Over-caution check: {wrongly_refused}/{len(yes_rows)} 'answerable' questions mein LLM ne galti se 'not found' bol diya")

Hallucination check: 2/6 'not available' questions mein LLM ne sahi refuse kiya
Over-caution check: 0/21 'answerable' questions mein LLM ne galti se 'not found' bol diya


In [44]:
!pip install -q pymupdf python-docx pandas sentence-transformers faiss-cpu groq

from google.colab import drive, userdata
drive.mount('/content/drive')

import sys, os
from pathlib import Path
PROJECT_DIR = Path("/content/drive/MyDrive/shifa-knowledge-assistant")
sys.path.append(str(PROJECT_DIR))
os.chdir(PROJECT_DIR)

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
print("Groq key loaded:", "✅ Yes" if GROQ_API_KEY else "❌ No")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Groq key loaded: ✅ Yes


In [45]:
%%writefile /content/drive/MyDrive/shifa-knowledge-assistant/rag_pipeline.py
"""
rag_pipeline.py  (Phase 8 + Phase 9)
Retrieved chunks ko context mein badalta hai, strict grounding instruction ke saath
Groq LLM ko bhejta hai, jawaab wapas deta hai, aur sirf un sources ki structured
citation list banata hai jo LLM ne asal mein apne jawaab mein use kiye.
"""

import os
import re
from groq import Groq

from retriever import load_knowledge_base, retrieve

GROQ_MODEL = "openai/gpt-oss-20b"

SYSTEM_PROMPT = """You are SHIFA Knowledge Assistant, a policy and procedure assistant for hospital staff at SHIFA General Hospital.

STRICT RULES:
1. Answer only from the provided hospital document context below. Do not use any outside knowledge.
2. If sufficient information is not available in the context, clearly say that the answer could not be found in the available documents. Do not guess or make up an answer.
3. When you use information from a source, refer to it using its tag, like [S1] or [S2]. Cite every factual claim.
4. This is a knowledge/policy assistant, not a medical diagnosis system. Do not give clinical advice beyond what is written in the policies.
5. Be concise and direct. Use short paragraphs or bullet points where helpful.
"""


def get_groq_client(api_key=None):
    api_key = api_key or os.environ.get("GROQ_API_KEY")
    if not api_key:
        raise ValueError("GROQ_API_KEY nahi mili. Isse environment variable ya argument se pass karo.")
    return Groq(api_key=api_key)


def build_context(chunks):
    """
    Retrieved chunks ko numbered source blocks mein badalta hai, jaise:
    [S1] Source: EM-01_Emergency_Admission_Policy.pdf (Emergency, page 1)
    <chunk text>

    Har chunk ko uska tag ("S1", "S2"...) bhi assign karke wapas deta hai, taaki
    baad mein extract_cited_sources() isi mapping se match kar sake.
    """
    blocks = []
    tagged_chunks = []
    for i, chunk in enumerate(chunks, start=1):
        tag = f"S{i}"
        header = f"[{tag}] Source: {chunk['filename']} ({chunk['department']}, page {chunk['page']})"
        blocks.append(f"{header}\n{chunk['text']}")
        chunk_with_tag = dict(chunk)
        chunk_with_tag["tag"] = tag
        tagged_chunks.append(chunk_with_tag)
    return "\n\n---\n\n".join(blocks), tagged_chunks


def extract_cited_sources(answer_text, tagged_chunks, excerpt_len=200):
    """
    Answer text mein se [S1], [S2] jaise tags dhoondhta hai, aur sirf unhi
    chunks ki structured citation list banata hai jo asal mein use hue.
    Agar LLM ne koi tag use hi nahi kiya (ya "not found" jawaab diya), khaali list aati hai.
    """
    used_tags = set(re.findall(r"\[S(\d+)\]", answer_text))   # jaise {"1", "3"}

    cited = []
    for chunk in tagged_chunks:
        tag_number = chunk["tag"].lstrip("S")
        if tag_number in used_tags:
            cited.append({
                "tag": chunk["tag"],
                "filename": chunk["filename"],
                "title": chunk["title"],
                "department": chunk["department"],
                "page": chunk["page"],
                "score": round(chunk["score"], 3),
                "excerpt": chunk["text"][:excerpt_len].strip() + ("..." if len(chunk["text"]) > excerpt_len else ""),
            })

    # Tag number ke hisaab se sort karo (S1, S2, S3...) taaki order saaf rahe
    cited.sort(key=lambda c: int(c["tag"].lstrip("S")))
    return cited


def answer_question(question, index, chunks_db, client, top_k=4, temperature=0.1):
    """
    Poora RAG flow: question -> retrieve -> Groq -> grounded answer -> cited sources.

    Return karta hai:
    {
        "answer": str,                 # LLM ka jawaab, [S1] tags ke saath
        "cited_sources": list,         # sirf wo sources jo jawaab mein use hue
        "all_retrieved": list,         # debugging ke liye, sab retrieved chunks
    }
    """
    # 1. Retrieve
    retrieved = retrieve(question, index, chunks_db, top_k=top_k)

    if not retrieved:
        return {
            "answer": "The answer could not be found in the available documents.",
            "cited_sources": [],
            "all_retrieved": [],
        }

    # 2. Context banao (tags ke saath)
    context_text, tagged_chunks = build_context(retrieved)

    # 3. Groq ko prompt bhejo
    user_message = f"""Context from hospital documents:

{context_text}

---

Staff question: {question}

Answer the question using only the context above. Cite sources using their tags like [S1], [S2]."""

    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
        temperature=temperature,
        max_tokens=600,
    )

    answer_text = response.choices[0].message.content

    # 4. Sirf cited sources nikaalo
    cited_sources = extract_cited_sources(answer_text, tagged_chunks)

    return {
        "answer": answer_text,
        "cited_sources": cited_sources,
        "all_retrieved": tagged_chunks,
    }


def format_sources_display(cited_sources):
    """Citation list ko ek readable text block mein format karta hai (Streamlit se pehle quick-check ke liye)."""
    if not cited_sources:
        return "(No sources cited)"

    lines = []
    for s in cited_sources:
        lines.append(f"[{s['tag']}] {s['filename']} | {s['department']} | Page {s['page']} | score={s['score']}")
        lines.append(f"    \"{s['excerpt']}\"")
    return "\n".join(lines)


if __name__ == "__main__":
    index, chunks_db = load_knowledge_base()
    client = get_groq_client()
    result = answer_question("What is the deposit for ICU admission?", index, chunks_db, client)
    print(result["answer"])
    print("\nSources:")
    print(format_sources_display(result["cited_sources"]))

Overwriting /content/drive/MyDrive/shifa-knowledge-assistant/rag_pipeline.py


In [46]:
import importlib
import rag_pipeline
importlib.reload(rag_pipeline)
from rag_pipeline import get_groq_client, answer_question, format_sources_display
from retriever import load_knowledge_base

index, chunks_db = load_knowledge_base()
client = get_groq_client(GROQ_API_KEY)

result = answer_question("What is the deposit for ICU admission?", index, chunks_db, client)

print("ANSWER:")
print(result["answer"])
print("\nCITED SOURCES:")
print(format_sources_display(result["cited_sources"]))

ANSWER:
The deposit for an ICU admission is **PKR 100,000**【S1】【S4】.

CITED SOURCES:
(No sources cited)


In [47]:
result2 = answer_question("What is the monthly salary of a staff nurse at SHIFA?", index, chunks_db, client)

print("ANSWER:")
print(result2["answer"])
print("\nCITED SOURCES:")
print(format_sources_display(result2["cited_sources"]))

ANSWER:
I’m sorry, but the available documents do not contain information about the monthly salary of a staff nurse at SHIFA General Hospital.

CITED SOURCES:
(No sources cited)


In [48]:
result3 = answer_question("What is the procedure for admitting an emergency patient?", index, chunks_db, client)

print("ANSWER:")
print(result3["answer"])
print("\nCITED SOURCES:")
print(format_sources_display(result3["cited_sources"]))
print(f"\n(Retrieved {len(result3['all_retrieved'])} chunks total, {len(result3['cited_sources'])} actually cited)")

ANSWER:
**Procedure for admitting an emergency patient**

1. **Arrival & triage** – Patient is seen by a triage nurse within 5 min and assigned a triage category (1‑5) [S1].  
2. **Quick registration** – An emergency MRN is created with minimal data (name/alias, approximate age, sex, mode of arrival, relative’s contact). Full registration under ADM‑01 is completed within 24 h by the Admissions Desk [S1], [S2].  
3. **Doctor assessment** – The ED doctor evaluates the patient within the time target set for the triage category [S1].  
4. **Consent** – Written consent is obtained from the patient or guardian; if unavailable, two physicians (one senior) document emergency consent before life‑saving treatment [S2].  
5. **Disposition decision** – The ED physician decides on discharge, ED observation (≤6 h), or inpatient admission. If inpatient admission is chosen, the patient is transferred to the appropriate ward following the hospital’s bed‑management procedures [S1], [S2].  

These steps 

In [49]:
import pandas as pd

eval_df = pd.read_csv(PROJECT_DIR / "evaluation_questions.csv")
sample = eval_df.sample(5, random_state=42)   # pura 26 chalane ki zaroorat nahi, 5 kaafi hain quick check ke liye

for _, row in sample.iterrows():
    result = answer_question(row["question"], index, chunks_db, client)
    print(f"Q: {row['question']}")
    print(f"A: {result['answer'][:200]}...")
    print(f"Sources: {[s['filename'] for s in result['cited_sources']]}")
    print()

Q: How long should alcohol-based hand rub be rubbed on the hands?
A: Alcohol‑based hand rub should be rubbed on all surfaces of the hands for **20 to 30 seconds** until the hands are dry. [S1] [S2]...
Sources: ['PS-02_Infection_Prevention_Guidelines.docx', 'PS-02_Infection_Prevention_Guidelines.docx']

Q: Which medicines are classified as high-alert medications?
A: High‑alert medications at SHIFA General Hospital are:

- Insulin  
- Heparin  
- Concentrated potassium chloride  
- Magnesium sulfate  
- Opioids  
- Chemotherapy agents  
- Neuromuscular blocking ag...
Sources: []

Q: What should I do after a needle-stick injury?
A: **After a needle‑stick injury:**

1. **Clean the wound**  
   - Let the wound bleed.  
   - Wash with soap and running water for 40–60 seconds (do not squeeze or scrub).  
   - If the injury involved ...
Sources: ['PS-02_Infection_Prevention_Guidelines.docx']

Q: What is the monthly salary of a staff nurse at SHIFA?
A: I couldn’t find any information on the mon

# PHASE 10

In [50]:
%%writefile /content/drive/MyDrive/shifa-knowledge-assistant/app.py
"""
app.py  (Phase 10)
SHIFA Knowledge Assistant - Streamlit chat UI.

IMPORTANT: Ye app documents ko kabhi re-process/re-embed NAHI karta.
Ye sirf PRE-BUILT knowledge_base/ (faiss.index + chunks.pkl) ko LOAD karta hai,
jo build_index.py se Colab mein pehle hi bana chuka hai.
"""

import streamlit as st

from retriever import load_knowledge_base
from rag_pipeline import get_groq_client, answer_question
from embedder import get_model as load_embedding_model


# ---------------------------------------------------------------
# Page config (sabse pehle call hona chahiye)
# ---------------------------------------------------------------
st.set_page_config(
    page_title="SHIFA Knowledge Assistant",
    page_icon="🏥",
    layout="wide",
)


# ---------------------------------------------------------------
# Cached loaders - ye sirf EK BAAR chalte hain, har interaction par nahi
# ---------------------------------------------------------------
@st.cache_resource
def get_knowledge_base():
    """Saved FAISS index + chunks load karta hai. Documents re-process NAHI hote."""
    return load_knowledge_base()


@st.cache_resource
def get_embedding_model():
    """all-MiniLM-L6-v2 model ek baar load karke cache mein rakhta hai."""
    return load_embedding_model()


@st.cache_resource
def get_client():
    """Groq client banata hai, API key Streamlit secrets se leta hai."""
    api_key = st.secrets.get("GROQ_API_KEY")
    if not api_key:
        st.error("GROQ_API_KEY nahi mili. .streamlit/secrets.toml mein daalo (Colab/local) "
                  "ya Streamlit Cloud ke Secrets settings mein daalo.")
        st.stop()
    return get_groq_client(api_key)


# ---------------------------------------------------------------
# App start: knowledge base + model + client load karo
# ---------------------------------------------------------------
with st.spinner("Knowledge base load ho raha hai..."):
    index, chunks_db = get_knowledge_base()
    get_embedding_model()          # isse load karke cache mein daal do, retriever isko use karega
    client = get_client()


# ---------------------------------------------------------------
# Sidebar: hospital info aur disclaimer
# ---------------------------------------------------------------
with st.sidebar:
    st.title("🏥 SHIFA Knowledge Assistant")
    st.caption("Enterprise RAG Assistant for Hospital Staff")

    st.warning("⚠️ DEMO / SYNTHETIC HOSPITAL DATA — NOT FOR REAL CLINICAL USE", icon="⚠️")

    st.markdown("---")
    st.markdown(f"**Knowledge base:**")
    st.markdown(f"- {len(chunks_db)} chunks indexed")
    departments = sorted({c["department"] for c in chunks_db})
    st.markdown(f"- Departments: {', '.join(departments)}")

    st.markdown("---")
    st.markdown("**Example questions:**")
    example_questions = [
        "What is the procedure for admitting an emergency patient?",
        "What is the deposit for ICU admission?",
        "What should I do after a needle-stick injury?",
        "How long do insulin doses need independent double-check?",
    ]
    for q in example_questions:
        if st.button(q, use_container_width=True):
            st.session_state["pending_question"] = q

    st.markdown("---")
    if st.button("🗑️ Clear chat", use_container_width=True):
        st.session_state["messages"] = []
        st.rerun()


# ---------------------------------------------------------------
# Chat state
# ---------------------------------------------------------------
if "messages" not in st.session_state:
    st.session_state["messages"] = []

st.title("Ask SHIFA Knowledge Assistant")
st.caption("Ask a question about hospital policies and procedures. Answers are grounded in official documents only.")


# ---------------------------------------------------------------
# Purana chat history dikhao
# ---------------------------------------------------------------
for msg in st.session_state["messages"]:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])
        if msg["role"] == "assistant" and msg.get("sources"):
            with st.expander(f"📚 Sources ({len(msg['sources'])})"):
                for s in msg["sources"]:
                    st.markdown(f"**[{s['tag']}] {s['filename']}** — {s['department']}, Page {s['page']} "
                                f"(relevance: {s['score']:.2f})")
                    st.caption(f"\"{s['excerpt']}\"")


# ---------------------------------------------------------------
# Naya sawaal lo (ya sidebar se example click hua ho)
# ---------------------------------------------------------------
question = st.chat_input("Type your question here...")

if "pending_question" in st.session_state:
    question = st.session_state.pop("pending_question")

if question:
    # User ka message dikhao + save karo
    st.session_state["messages"].append({"role": "user", "content": question})
    with st.chat_message("user"):
        st.markdown(question)

    # Jawaab generate karo
    with st.chat_message("assistant"):
        with st.spinner("Documents search ho rahe hain..."):
            result = answer_question(question, index, chunks_db, client, top_k=4)

        st.markdown(result["answer"])

        if result["cited_sources"]:
            with st.expander(f"📚 Sources ({len(result['cited_sources'])})"):
                for s in result["cited_sources"]:
                    st.markdown(f"**[{s['tag']}] {s['filename']}** — {s['department']}, Page {s['page']} "
                                f"(relevance: {s['score']:.2f})")
                    st.caption(f"\"{s['excerpt']}\"")

    # Assistant ka message save karo
    st.session_state["messages"].append({
        "role": "assistant",
        "content": result["answer"],
        "sources": result["cited_sources"],
    })

Overwriting /content/drive/MyDrive/shifa-knowledge-assistant/app.py


In [51]:
import os

secrets_dir = PROJECT_DIR / ".streamlit"
secrets_dir.mkdir(exist_ok=True)

secrets_content = f'GROQ_API_KEY = "{GROQ_API_KEY}"\n'
(secrets_dir / "secrets.toml").write_text(secrets_content)

print("✅ .streamlit/secrets.toml bana diya")
print("Dhyan rahe: ye file .gitignore mein hai, isliye GitHub par kabhi upload nahi hogi.")

✅ .streamlit/secrets.toml bana diya
Dhyan rahe: ye file .gitignore mein hai, isliye GitHub par kabhi upload nahi hogi.
